
# Study-aligned Data Acquisition and Cleaning Pipeline for Gold and Silver Futures

This notebook updates the original acquisition/cleaning script into a **study-aligned pipeline** with stricter handling of missing data and trading dates.

## What changed
1. **Each raw dataset is cleaned separately**  
   - sort chronologically  
   - remove duplicate dates  
   - inspect shape, missing values, and invalid rows  

2. **Low-frequency variables are handled differently from market-traded series**  
   - **Monthly / quarterly / policy variables** are aligned to a common daily timeline and **interpolated over time**
   - **Market-traded series are _not_ blindly interpolated**
   - missing traded dates are audited against the **COMEX calendar** and can be optionally backfilled from **Yahoo Finance**

3. **The final modeling panel is restricted to valid target trading dates**  
   - since the target is **gold / silver futures**, the final panel is restricted to **COMEX sessions**

4. **The old structural issues are removed**  
   - no undefined `financial_df`
   - no undefined `economic_day`
   - no saving of undefined `*_TRUE` / `*_MIX` datasets

---


In [ ]:

# Core dependencies
!pip -q install pandas_datareader fredapi exchange-calendars openpyxl xlrd yfinance


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 213.3/213.3 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.1/58.1 kB 2.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ibis-framework 9.5.0 requires toolz<1,>=0.11, but you have toolz 1.1.0 which is incompatible.


In [ ]:

# Google Drive mount (Colab only)
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception:
    print("Drive mount skipped (not running in Google Colab).")


Mounted at /content/drive


In [ ]:

import os
import warnings
from datetime import datetime, timedelta

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from fredapi import Fred
import exchange_calendars as xcals

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 140)



## Configuration

Update the paths below if your folder structure is different.


In [ ]:
# =========================
# Global configuration
# =========================
start_date = datetime(2015, 4, 1)
end_date   = datetime(2025, 4, 30)

BASE_DIR = "/content/drive/MyDrive/ResearchTestingGold/Silver gold Research/Silver gold"
FINANCIAL_DIR = os.path.join(BASE_DIR, "data", "investing.comData")
GEPU_FILE = os.path.join(BASE_DIR, "data", "Global_Policy_Uncertainty_Data.xlsx")
GPR_FILE  = os.path.join(BASE_DIR, "data", "data_gpr_daily_recent.xls")
OUTPUT_DIR = os.path.join(BASE_DIR, "CleanDATA_2026_study_aligned")

os.makedirs(OUTPUT_DIR, exist_ok=True)

TARGET_CALENDAR_NAME = "COMEX"
USE_YAHOO_BACKFILL = True    # backfill unresolved traded-session gaps from Yahoo Finance
YAHOO_PROGRESS = False
YAHOO_AUTO_ADJUST = False
SAVE_BACKFILL_ARTIFACTS = True

# Manual holiday overrides for dates confirmed by CME holiday notices but not surfaced
# by exchange_calendars metadata in some environments.
COMEX_HOLIDAY_MANUAL_OVERRIDES = pd.DataFrame({
    "Date": pd.to_datetime([
        "2022-06-20",
        "2023-06-19",
        "2024-06-19",
    ]),
    "holiday_name": [
        "Juneteenth National Independence Day",
        "Juneteenth National Independence Day",
        "Juneteenth National Independence Day",
    ],
    "holiday_type": [
        "manual_override",
        "manual_override",
        "manual_override",
    ],
    "session_effect": [
        "holiday_schedule_notice",
        "holiday_schedule_notice",
        "holiday_schedule_notice",
    ],
    "close_time": [pd.NaT, pd.NaT, pd.NaT],
    "open_time": [pd.NaT, pd.NaT, pd.NaT],
    "tag_source": [
        "manual_cme_notice",
        "manual_cme_notice",
        "manual_cme_notice",
    ],
    "source_note": [
        "Juneteenth holiday schedule manually added because exchange_calendars did not surface metadata for this date.",
        "Juneteenth holiday schedule manually added because exchange_calendars did not surface metadata for this date.",
        "Juneteenth holiday schedule manually added because exchange_calendars did not surface metadata for this date.",
    ],
})


# Series-specific missing-date overrides for non-COMEX markets that still appear in the
# multivariate panel. These are used only when explaining missing observations for a
# specific series; they do not change the COMEX holiday feature columns.
SERIES_SPECIFIC_MISSING_DATE_OVERRIDES = pd.DataFrame({
    "Date": pd.to_datetime([
        "2016-11-11",
    ]),
    "series": [
        "UST10Y_Treasury_Yield",
    ],
    "holiday_name": [
        "Veterans Day",
    ],
    "holiday_type": [
        "manual_override",
    ],
    "session_effect": [
        "fixed_income_holiday",
    ],
    "close_time": [pd.NaT],
    "open_time": [pd.NaT],
    "tag_source": [
        "manual_sifma_notice",
    ],
    "source_note": [
        "SIFMA fixed-income market holiday; COMEX commodities treated this date as a normal day.",
    ],
})

HOLIDAY_FEATURE_COLS = [
    "comex_holiday_related",
    "comex_full_closure_holiday",
    "comex_special_close",
    "comex_special_open",
    "is_pre_holiday",
    "is_post_holiday",
]

# Use an environment variable when possible:
import os; os.environ["FRED_API_KEY"] = "cd5c7da352a024ed73baeed1c751ba38"
fred_api_key = os.environ.get("FRED_API_KEY", "")
if not fred_api_key:
    print("WARNING: FRED_API_KEY is empty. Set it before running FRED download cells.")

financial_files = {
    "Gold_Futures": "Gold Futures Historical Data.csv",
    "Silver_Futures": "Silver Futures Historical Data.csv",
    "Crude_Oil_Futures": "Crude Oil WTI Futures Historical Data.csv",
    "Copper_Futures": "Copper Futures Historical Data.csv",
    "Platinum_Futures": "Platinum Futures Historical Data.csv",
    "Palladium_Futures": "Palladium Futures Historical Data.csv",
    "iShares_MSCI_World_ETF": "MSCI Stock Price History.csv",
    "NASDAQ_100": "Nasdaq 100 Historical Data.csv",
    "SnP500": "S&P 500 Historical Data.csv",
    "UST10Y_Treasury_Yield": "United States 10-Year Bond Yield Historical Data.csv",
    "CBOE_VIX": "CBOE Volatility Index Historical Data.csv",
    "US30":"US 30 Cash Historical Data.csv",
    "USD_index":"USD Index Historical Data.csv"
}

yahoo_tickers = {
    "Gold_Futures": "GC=F",
    "Silver_Futures": "SI=F",
    "Crude_Oil_Futures": "CL=F",
    "Copper_Futures": "HG=F",
    "Platinum_Futures": "PL=F",
    "Palladium_Futures": "PA=F",
    "iShares_MSCI_World_ETF": "URTH",
    "NASDAQ_100": "^NDX",
    "SnP500": "^GSPC",
    "UST10Y_Treasury_Yield": "^TNX",
    "CBOE_VIX": "^VIX",
    "US30":"^DJI",
    "USD_index":"DX-Y.NYB"
}

# Existing literature-based feature sets from the original code
silver_features_RRL = [
    "Gold_Futures",
    "US30",
    "SnP500",
    "NASDAQ_100",
    "USD_index",
]

gold_features_RRL = [
    "Silver_Futures",
    "Crude_Oil_Futures",
    "UST10Y_Treasury_Yield",
    "Federal_Funds_Rate",
    "Employment_Pop_Ratio",
    "gepu",
    "gpr_daily",
]

target_gold = "Gold_Futures"
target_silver = "Silver_Futures"


In [ ]:
# =========================
# Helper functions
# =========================
def to_numeric_series(series, is_percent=False):
    s = series.astype(str).str.strip()
    s = s.replace({
        "": np.nan,
        "nan": np.nan,
        "None": np.nan,
        "-": np.nan,
        "—": np.nan,
    })
    if is_percent:
        s = s.str.replace("%", "", regex=False)
    s = s.str.replace(",", "", regex=False)
    return pd.to_numeric(s, errors="coerce")

def parse_volume_series(series):
    def _convert(x):
        if pd.isna(x):
            return np.nan
        x = str(x).strip().upper().replace(",", "")
        if x in {"", "NAN", "-", "—"}:
            return np.nan
        mult = 1.0
        if x.endswith("K"):
            mult = 1_000.0
            x = x[:-1]
        elif x.endswith("M"):
            mult = 1_000_000.0
            x = x[:-1]
        elif x.endswith("B"):
            mult = 1_000_000_000.0
            x = x[:-1]
        try:
            return float(x) * mult
        except Exception:
            return np.nan
    return series.apply(_convert)

def clean_investing_price_frame(df_raw, series_name, start_date, end_date):
    df = df_raw.copy()
    df.columns = [c.strip() for c in df.columns]

    if "Date" not in df.columns or "Price" not in df.columns:
        raise ValueError(f"{series_name}: expected 'Date' and 'Price' columns.")

    raw_rows = len(df)

    # Parse dates
    df["Date"] = pd.to_datetime(df["Date"], errors="coerce")

    # Parse numeric columns when present
    numeric_cols = ["Price", "Open", "High", "Low"]
    for col in numeric_cols:
        if col in df.columns:
            df[col] = to_numeric_series(df[col])

    if "Change %" in df.columns:
        df["Change %"] = to_numeric_series(df["Change %"], is_percent=True)

    if "Vol." in df.columns:
        df["Vol."] = parse_volume_series(df["Vol."])

    # Invalid / out-of-range rows
    invalid_mask = df["Date"].isna() | df["Price"].isna()
    out_of_range_mask = (~df["Date"].isna()) & (
        (df["Date"] < pd.Timestamp(start_date)) | (df["Date"] > pd.Timestamp(end_date))
    )

    invalid_rows = int(invalid_mask.sum())
    out_of_range_rows = int(out_of_range_mask.sum())

    df = df.loc[~(invalid_mask | out_of_range_mask)].copy()

    duplicates_before = int(df.duplicated(subset=["Date"]).sum())

    df = (
        df.sort_values("Date")
          .drop_duplicates(subset=["Date"], keep="last")
          .reset_index(drop=True)
    )

    cleaned_rows = len(df)

    quality = {
        "dataset": series_name,
        "raw_rows": raw_rows,
        "cleaned_rows": cleaned_rows,
        "duplicates_removed": duplicates_before,
        "invalid_rows_removed": invalid_rows,
        "out_of_range_rows_removed": out_of_range_rows,
        "price_missing_after_clean": int(df["Price"].isna().sum()),
        "date_min": df["Date"].min(),
        "date_max": df["Date"].max(),
    }

    price_df = df[["Date", "Price"]].rename(columns={"Price": series_name}).set_index("Date")
    return df, price_df, quality

def fetch_fred_dataframe(series_dict, start_date, end_date, api_key):
    if not api_key:
        raise ValueError("FRED_API_KEY is empty. Set os.environ['FRED_API_KEY'] first.")
    fred = Fred(api_key=api_key)
    frames = []
    for col_name, series_id in series_dict.items():
        series = fred.get_series(
            series_id,
            observation_start=start_date,
            observation_end=end_date,
        )
        frames.append(series.rename(col_name).to_frame())
    out = pd.concat(frames, axis=1).sort_index()
    out.index = pd.to_datetime(out.index)
    return out

def align_to_daily_with_interpolation(df, daily_index):
    out = df.copy()
    out.index = pd.to_datetime(out.index)
    out = out.sort_index().reindex(daily_index)
    out = out.interpolate(method="time", limit_direction="both")
    return out

def summarize_frame(df, name):
    if len(df) == 0:
        return pd.DataFrame([{
            "dataset": name,
            "rows": 0,
            "cols": 0,
            "date_min": pd.NaT,
            "date_max": pd.NaT,
            "duplicate_index": np.nan,
            "total_missing_cells": np.nan,
        }])
    idx = pd.to_datetime(df.index) if not isinstance(df.index, pd.RangeIndex) else pd.to_datetime(df["Date"])
    return pd.DataFrame([{
        "dataset": name,
        "rows": len(df),
        "cols": df.shape[1],
        "date_min": idx.min(),
        "date_max": idx.max(),
        "duplicate_index": int(pd.Index(idx).duplicated().sum()),
        "total_missing_cells": int(df.isna().sum().sum()),
    }])

def audit_market_missing_vs_calendar(price_df, target_sessions):
    target_sessions = pd.DatetimeIndex(target_sessions)
    reports = []
    unresolved_blocks = []

    for col in price_df.columns:
        observed = pd.DatetimeIndex(price_df[col].dropna().index).normalize()
        observed = observed.unique().sort_values()

        missing_on_target_sessions = target_sessions.difference(observed)

        reports.append({
            "series": col,
            "observed_points": len(observed),
            "target_sessions": len(target_sessions),
            "missing_on_target_sessions": len(missing_on_target_sessions),
            "first_observed": observed.min() if len(observed) else pd.NaT,
            "last_observed": observed.max() if len(observed) else pd.NaT,
        })

        if len(missing_on_target_sessions):
            unresolved_blocks.append(
                pd.DataFrame({
                    "series": col,
                    "date": missing_on_target_sessions
                })
            )

    report_df = pd.DataFrame(reports).sort_values(["missing_on_target_sessions", "series"], ascending=[False, True])
    unresolved_df = (
        pd.concat(unresolved_blocks, ignore_index=True).sort_values(["series", "date"])
        if unresolved_blocks else
        pd.DataFrame(columns=["series", "date"])
    )
    return report_df, unresolved_df

def fetch_yahoo_close(ticker, start_date, end_date, progress=False, auto_adjust=False):
    import yfinance as yf

    df = yf.download(
        ticker,
        start=pd.Timestamp(start_date).strftime("%Y-%m-%d"),
        end=(pd.Timestamp(end_date) + pd.Timedelta(days=1)).strftime("%Y-%m-%d"),
        auto_adjust=auto_adjust,
        progress=progress,
    )

    if df.empty:
        return pd.Series(dtype=float)

    if isinstance(df.columns, pd.MultiIndex):
        if ("Close", ticker) in df.columns:
            close = df[("Close", ticker)]
        else:
            close = df["Close"].iloc[:, 0]
    else:
        close = df["Close"]

    close.index = pd.to_datetime(close.index).normalize()
    close.name = ticker
    return pd.to_numeric(close, errors="coerce")

def yahoo_backfill_unresolved_only(
    price_df,
    unresolved_df,
    ticker_map,
    start_date,
    end_date,
    progress=False,
    auto_adjust=False,
):
    updated = price_df.copy().sort_index()
    detail_rows = []
    summary_rows = []

    if unresolved_df.empty:
        return (
            updated,
            pd.DataFrame(columns=["series", "ticker", "unresolved_before", "filled_count", "unresolved_after", "status"]),
            pd.DataFrame(columns=["series", "date", "ticker", "yahoo_close", "status"]),
        )

    unresolved_df = unresolved_df.copy()
    unresolved_df["date"] = pd.to_datetime(unresolved_df["date"]).dt.normalize()

    for series_name, grp in unresolved_df.groupby("series"):
        missing_dates = pd.DatetimeIndex(sorted(pd.to_datetime(grp["date"]).unique()))
        ticker = ticker_map.get(series_name)

        if ticker is None:
            summary_rows.append({
                "series": series_name,
                "ticker": None,
                "unresolved_before": len(missing_dates),
                "filled_count": 0,
                "unresolved_after": len(missing_dates),
                "status": "no_ticker_mapping",
            })
            for dt in missing_dates:
                detail_rows.append({
                    "series": series_name,
                    "date": dt,
                    "ticker": None,
                    "yahoo_close": np.nan,
                    "status": "no_ticker_mapping",
                })
            continue

        yahoo_close = fetch_yahoo_close(
            ticker=ticker,
            start_date=start_date,
            end_date=end_date,
            progress=progress,
            auto_adjust=auto_adjust,
        )

        if yahoo_close.empty:
            summary_rows.append({
                "series": series_name,
                "ticker": ticker,
                "unresolved_before": len(missing_dates),
                "filled_count": 0,
                "unresolved_after": len(missing_dates),
                "status": "no_yahoo_data",
            })
            for dt in missing_dates:
                detail_rows.append({
                    "series": series_name,
                    "date": dt,
                    "ticker": ticker,
                    "yahoo_close": np.nan,
                    "status": "no_yahoo_data",
                })
            continue

        updated = updated.reindex(updated.index.union(missing_dates))
        yahoo_on_missing = yahoo_close.reindex(missing_dates)

        fill_dates = yahoo_on_missing[yahoo_on_missing.notna()].index
        unresolved_after_dates = yahoo_on_missing[yahoo_on_missing.isna()].index

        if len(fill_dates):
            updated.loc[fill_dates, series_name] = yahoo_on_missing.loc[fill_dates].values

        for dt in fill_dates:
            detail_rows.append({
                "series": series_name,
                "date": dt,
                "ticker": ticker,
                "yahoo_close": float(yahoo_on_missing.loc[dt]),
                "status": "filled_from_yahoo",
            })

        for dt in unresolved_after_dates:
            detail_rows.append({
                "series": series_name,
                "date": dt,
                "ticker": ticker,
                "yahoo_close": np.nan,
                "status": "still_unresolved_after_yahoo",
            })

        summary_rows.append({
            "series": series_name,
            "ticker": ticker,
            "unresolved_before": len(missing_dates),
            "filled_count": len(fill_dates),
            "unresolved_after": len(unresolved_after_dates),
            "status": "ok" if len(fill_dates) else "no_fill_found",
        })

    updated = updated.sort_index()
    detail_df = pd.DataFrame(detail_rows).sort_values(["series", "date"]).reset_index(drop=True)
    summary_df = pd.DataFrame(summary_rows).sort_values(["unresolved_before", "series"], ascending=[False, True]).reset_index(drop=True)
    return updated, summary_df, detail_df

def _empty_holiday_frame():
    return pd.DataFrame(
        columns=[
            "Date",
            "holiday_name",
            "holiday_type",
            "session_effect",
            "close_time",
            "open_time",
            "tag_source",
            "source_note",
        ]
    )

def _normalize_manual_holiday_overrides(manual_overrides, start_date, end_date):
    if manual_overrides is None or len(manual_overrides) == 0:
        return _empty_holiday_frame()

    out = manual_overrides.copy()
    out["Date"] = pd.to_datetime(out["Date"]).dt.normalize()

    default_values = {
        "holiday_name": "Manual holiday override",
        "holiday_type": "manual_override",
        "session_effect": "manual_override",
        "close_time": pd.NaT,
        "open_time": pd.NaT,
        "tag_source": "manual_override",
        "source_note": "Manual holiday override",
    }
    for col, default_val in default_values.items():
        if col not in out.columns:
            out[col] = default_val

    start = pd.Timestamp(start_date).normalize()
    end = pd.Timestamp(end_date).normalize()

    out = out.loc[(out["Date"] >= start) & (out["Date"] <= end)].copy()
    out = out[
        ["Date", "holiday_name", "holiday_type", "session_effect", "close_time", "open_time", "tag_source", "source_note"]
    ].sort_values(["Date", "holiday_name"]).drop_duplicates().reset_index(drop=True)
    return out

def build_calendar_holiday_tables(calendar, start_date, end_date, manual_overrides=None):
    start = pd.Timestamp(start_date).normalize()
    end = pd.Timestamp(end_date).normalize()

    def _frames_from_rules(rules, holiday_type, session_effect, close_time=pd.NaT, open_time=pd.NaT):
        frames = []
        for rule in rules:
            rule_dates = pd.DatetimeIndex(rule.dates(start, end))
            if len(rule_dates) == 0:
                continue
            frames.append(pd.DataFrame({
                "Date": rule_dates,
                "holiday_name": getattr(rule, "name", str(rule)),
                "holiday_type": holiday_type,
                "session_effect": session_effect,
                "close_time": close_time,
                "open_time": open_time,
                "tag_source": "exchange_calendars",
                "source_note": "Tagged from exchange_calendars holiday rules.",
            }))
        return frames

    full_frames = []
    regular_holiday_rules = getattr(getattr(calendar, "regular_holidays", None), "rules", [])
    full_frames.extend(
        _frames_from_rules(
            regular_holiday_rules,
            holiday_type="full_closure_regular",
            session_effect="closed",
        )
    )

    adhoc_dates = pd.DatetimeIndex(getattr(calendar, "adhoc_holidays", []))
    if len(adhoc_dates) > 0:
        adhoc_dates = adhoc_dates[(adhoc_dates >= start) & (adhoc_dates <= end)]
        if len(adhoc_dates) > 0:
            full_frames.append(pd.DataFrame({
                "Date": adhoc_dates,
                "holiday_name": "Ad hoc full-closure holiday",
                "holiday_type": "full_closure_adhoc",
                "session_effect": "closed",
                "close_time": pd.NaT,
                "open_time": pd.NaT,
                "tag_source": "exchange_calendars",
                "source_note": "Tagged from exchange_calendars ad hoc holiday list.",
            }))

    comex_full_closure_holidays = (
        pd.concat(full_frames, ignore_index=True)
        if full_frames else
        _empty_holiday_frame()
    )
    if not comex_full_closure_holidays.empty:
        comex_full_closure_holidays = (
            comex_full_closure_holidays
            .sort_values(["Date", "holiday_type", "holiday_name"])
            .drop_duplicates(subset=["Date", "holiday_type", "holiday_name"])
            .reset_index(drop=True)
        )

    def _special_frames(special_rules, session_effect):
        frames = []
        for tm, holiday_calendar in special_rules:
            rules = getattr(holiday_calendar, "rules", [])
            if rules:
                frames.extend(
                    _frames_from_rules(
                        rules,
                        holiday_type=session_effect,
                        session_effect=session_effect,
                        close_time=str(tm) if session_effect == "special_close" else pd.NaT,
                        open_time=str(tm) if session_effect == "special_open" else pd.NaT,
                    )
                )
            else:
                cal_dates = pd.DatetimeIndex(holiday_calendar.holidays(start, end))
                if len(cal_dates) == 0:
                    continue
                frames.append(pd.DataFrame({
                    "Date": cal_dates,
                    "holiday_name": f"{session_effect} holiday",
                    "holiday_type": session_effect,
                    "session_effect": session_effect,
                    "close_time": str(tm) if session_effect == "special_close" else pd.NaT,
                    "open_time": str(tm) if session_effect == "special_open" else pd.NaT,
                    "tag_source": "exchange_calendars",
                    "source_note": "Tagged from exchange_calendars special session rules.",
                }))
        return frames

    special_close_frames = _special_frames(getattr(calendar, "special_closes", []), "special_close")
    special_close_adhoc = getattr(calendar, "special_closes_adhoc", [])
    for tm, dates in special_close_adhoc:
        dt_idx = pd.DatetimeIndex(dates)
        dt_idx = dt_idx[(dt_idx >= start) & (dt_idx <= end)]
        if len(dt_idx) == 0:
            continue
        special_close_frames.append(pd.DataFrame({
            "Date": dt_idx,
            "holiday_name": "Ad hoc special close",
            "holiday_type": "special_close_adhoc",
            "session_effect": "special_close",
            "close_time": str(tm),
            "open_time": pd.NaT,
            "tag_source": "exchange_calendars",
            "source_note": "Tagged from exchange_calendars ad hoc special closes.",
        }))
    comex_special_closes = (
        pd.concat(special_close_frames, ignore_index=True)
        if special_close_frames else
        _empty_holiday_frame()
    )
    if not comex_special_closes.empty:
        comex_special_closes = (
            comex_special_closes
            .sort_values(["Date", "holiday_name", "close_time"])
            .drop_duplicates(subset=["Date", "holiday_name", "close_time"])
            .reset_index(drop=True)
        )

    special_open_frames = _special_frames(getattr(calendar, "special_opens", []), "special_open")
    special_open_adhoc = getattr(calendar, "special_opens_adhoc", [])
    for tm, dates in special_open_adhoc:
        dt_idx = pd.DatetimeIndex(dates)
        dt_idx = dt_idx[(dt_idx >= start) & (dt_idx <= end)]
        if len(dt_idx) == 0:
            continue
        special_open_frames.append(pd.DataFrame({
            "Date": dt_idx,
            "holiday_name": "Ad hoc special open",
            "holiday_type": "special_open_adhoc",
            "session_effect": "special_open",
            "close_time": pd.NaT,
            "open_time": str(tm),
            "tag_source": "exchange_calendars",
            "source_note": "Tagged from exchange_calendars ad hoc special opens.",
        }))
    comex_special_opens = (
        pd.concat(special_open_frames, ignore_index=True)
        if special_open_frames else
        _empty_holiday_frame()
    )
    if not comex_special_opens.empty:
        comex_special_opens = (
            comex_special_opens
            .sort_values(["Date", "holiday_name", "open_time"])
            .drop_duplicates(subset=["Date", "holiday_name", "open_time"])
            .reset_index(drop=True)
        )

    comex_manual_holiday_overrides = _normalize_manual_holiday_overrides(
        manual_overrides=manual_overrides,
        start_date=start_date,
        end_date=end_date,
    )

    comex_holiday_schedule = pd.concat(
        [
            comex_full_closure_holidays,
            comex_special_closes,
            comex_special_opens,
            comex_manual_holiday_overrides,
        ],
        ignore_index=True,
    )
    if not comex_holiday_schedule.empty:
        comex_holiday_schedule = (
            comex_holiday_schedule
            .sort_values(["Date", "session_effect", "holiday_name"])
            .drop_duplicates(subset=["Date", "session_effect", "holiday_name", "close_time", "open_time", "tag_source"])
            .reset_index(drop=True)
        )

    comex_holiday_dates_only = (
        comex_holiday_schedule[["Date"]]
        .drop_duplicates()
        .sort_values("Date")
        .reset_index(drop=True)
    )

    return (
        comex_full_closure_holidays,
        comex_special_closes,
        comex_special_opens,
        comex_manual_holiday_overrides,
        comex_holiday_schedule,
        comex_holiday_dates_only,
    )

def annotate_missing_with_comex_holidays(
    missing_dates,
    holiday_schedule,
    series_name,
    series_specific_overrides=None,
):
    expected_cols = [
        "Date",
        "series",
        "holiday_name",
        "holiday_type",
        "session_effect",
        "close_time",
        "open_time",
        "tag_source",
        "source_note",
        "holiday_related",
        "reason",
    ]

    missing_df = pd.DataFrame({
        "Date": pd.to_datetime(pd.DatetimeIndex(missing_dates)).normalize(),
        "series": series_name,
    })

    if missing_df.empty:
        return pd.DataFrame(columns=expected_cols)

    base_schedule = holiday_schedule.copy()
    if not base_schedule.empty:
        base_schedule["Date"] = pd.to_datetime(base_schedule["Date"]).dt.normalize()

    out = missing_df.merge(base_schedule, on="Date", how="left")

    if series_specific_overrides is not None and len(series_specific_overrides) > 0:
        override_cols = [
            "Date",
            "holiday_name",
            "holiday_type",
            "session_effect",
            "close_time",
            "open_time",
            "tag_source",
            "source_note",
        ]
        override_df = series_specific_overrides.copy()
        override_df["Date"] = pd.to_datetime(override_df["Date"]).dt.normalize()
        if "series" in override_df.columns:
            override_df = override_df.loc[override_df["series"] == series_name].copy()
        override_df = override_df[override_cols].drop_duplicates()

        if not override_df.empty:
            out = out.merge(override_df, on="Date", how="left", suffixes=("", "_override"))
            for col in override_cols[1:]:
                override_col = f"{col}_override"
                if override_col in out.columns:
                    out[col] = out[col].combine_first(out[override_col])
            out = out.drop(columns=[c for c in out.columns if c.endswith("_override")])

    for col in expected_cols:
        if col not in out.columns:
            out[col] = pd.NA

    out["holiday_related"] = out["holiday_name"].notna()
    out["reason"] = np.where(
        out["holiday_related"],
        out["holiday_name"].astype(str) + " (" + out["session_effect"].astype(str) + ")",
        "Not tagged as COMEX holiday metadata",
    )
    return (
        out[expected_cols]
        .sort_values(["Date", "series", "session_effect", "holiday_name"])
        .reset_index(drop=True)
    )



## Step 1–2. Load and clean each raw financial dataset separately

For every raw financial file:
- sort chronologically
- remove duplicate dates
- inspect shape
- inspect missing values / invalid rows
- keep the cleaned **Price** series for downstream merging


In [ ]:

financial_raw = {}
financial_clean_full = {}
financial_price_series = {}
financial_quality_rows = []

for series_name, filename in financial_files.items():
    file_path = os.path.join(FINANCIAL_DIR, filename)
    raw_df = pd.read_csv(file_path)
    financial_raw[series_name] = raw_df

    cleaned_full_df, cleaned_price_df, quality = clean_investing_price_frame(
        raw_df,
        series_name=series_name,
        start_date=start_date,
        end_date=end_date
    )

    financial_clean_full[series_name] = cleaned_full_df
    financial_price_series[series_name] = cleaned_price_df
    financial_quality_rows.append(quality)

financial_quality_report = pd.DataFrame(financial_quality_rows).sort_values("dataset").reset_index(drop=True)

financial_df = pd.concat(financial_price_series.values(), axis=1).sort_index()
financial_df.index.name = "Date"

print("Financial panel shape:", financial_df.shape)
display(financial_quality_report)
display(financial_df.head())
display(financial_df.tail())


Financial panel shape: (3156, 13)


,dataset,raw_rows,cleaned_rows,duplicates_removed,invalid_rows_removed,out_of_range_rows_removed,price_missing_after_clean,date_min,date_max
0,CBOE_VIX,2570,2570,0,0,0,0,2015-04-01,2025-04-30
1,Copper_Futures,2619,2619,0,0,0,0,2015-04-01,2025-04-30
2,Crude_Oil_Futures,2651,2651,0,0,0,0,2015-04-01,2025-04-30
3,Gold_Futures,2238,2238,0,0,0,0,2015-04-01,2025-04-30
4,NASDAQ_100,2536,2536,0,0,0,0,2015-04-01,2025-04-30
5,Palladium_Futures,2857,2857,0,0,0,0,2015-04-01,2025-04-30
6,Platinum_Futures,3219,2908,0,0,311,0,2015-04-01,2025-04-30
7,Silver_Futures,2124,2124,0,0,0,0,2015-04-01,2025-04-30
8,SnP500,2536,2536,0,0,0,0,2015-04-01,2025-04-30
9,US30,2841,2841,0,0,0,0,2015-04-01,2025-04-30


,Gold_Futures,Silver_Futures,Crude_Oil_Futures,Copper_Futures,Platinum_Futures,Palladium_Futures,iShares_MSCI_World_ETF,NASDAQ_100,SnP500,UST10Y_Treasury_Yield,CBOE_VIX,US30,USD_index
Date,,,,,,,,,,,,,
2015-04-01,1208.2,17.059,50.09,2.750,1161.20,748.85,61.04,4311.26,2059.7,1.859,15.11,17698.2,NaN
2015-04-02,1200.9,16.701,49.14,2.736,1155.50,746.30,61.51,4316.01,2067.0,1.913,14.67,17763.2,NaN
2015-04-03,NaN,16.720,49.14,2.736,NaN,747.90,NaN,NaN,NaN,1.842,NaN,17676.0,NaN
2015-04-05,NaN,NaN,NaN,NaN,1174.85,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2015-04-06,1218.6,17.110,52.14,2.719,1174.90,768.80,62.12,4350.98,2080.6,1.899,14.74,17880.8,NaN


,Gold_Futures,Silver_Futures,Crude_Oil_Futures,Copper_Futures,Platinum_Futures,Palladium_Futures,iShares_MSCI_World_ETF,NASDAQ_100,SnP500,UST10Y_Treasury_Yield,CBOE_VIX,US30,USD_index
Date,,,,,,,,,,,,,
2025-04-25,NaN,NaN,63.02,4.8950,969.8,933.7,535.36,19432.56,5525.21,4.254,24.84,40126.5,99.58
2025-04-27,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,39977.5,NaN
2025-04-28,3333.0,33.165,62.05,4.8645,993.1,946.1,534.95,19427.29,5528.75,4.205,25.15,40255.0,99.01
2025-04-29,3333.6,33.577,60.42,4.8725,985.5,936.7,540.46,19544.95,5560.83,4.171,24.17,40520.0,99.14
2025-04-30,3319.1,32.828,58.21,4.6090,969.4,934.4,545.11,19571.02,5569.06,4.167,24.70,40798.4,99.66



## Step 1–2. Download and prepare macroeconomic and sentiment datasets

These are kept separate first, then aligned later.


In [ ]:
# FRED economic indicators
economic_month = {
    "Federal_Funds_Rate": "FEDFUNDS",
    "CPI": "CPIAUCSL",
    "Employment_Pop_Ratio": "EMRATIO",
    "Labour_Force_Participation_Rate": "CIVPART",
    "Unemployment_Rate": "UNRATE",
    "Personal_Savings_Rate": "PSAVERT",
}

economic_quarter = {
    "Federal_Debt_to_GDP": "GFDEGDQ188S",
}

# Additional FRED series to integrate through the pipeline
# Keep the exact series IDs as column names for consistency and traceability.
fred_extra_series = {
    "T10Y2Y": "T10Y2Y",  # 10-Year Treasury Constant Maturity Minus 2-Year Treasury Constant Maturity
    "IGREA": "IGREA",    # Index of Global Real Economic Activity
}

economic_month_df = fetch_fred_dataframe(economic_month, start_date, end_date, fred_api_key)
economic_quarter_df = fetch_fred_dataframe(economic_quarter, start_date, end_date, fred_api_key)
fred_extra_df = fetch_fred_dataframe(fred_extra_series, start_date, end_date, fred_api_key)

display(summarize_frame(economic_month_df, "economic_month_df"))
display(summarize_frame(economic_quarter_df, "economic_quarter_df"))
display(summarize_frame(fred_extra_df, "fred_extra_df"))

display(economic_month_df.head())
display(economic_quarter_df.head())
display(fred_extra_df.head())

,dataset,rows,cols,date_min,date_max,duplicate_index,total_missing_cells
0,economic_month_df,121,6,2015-04-01,2025-04-01,0,0


,dataset,rows,cols,date_min,date_max,duplicate_index,total_missing_cells
0,economic_quarter_df,41,1,2015-04-01,2025-04-01,0,0


,dataset,rows,cols,date_min,date_max,duplicate_index,total_missing_cells
0,fred_extra_df,2664,2,2015-04-01,2025-04-30,0,2685


,Federal_Funds_Rate,CPI,Employment_Pop_Ratio,Labour_Force_Participation_Rate,Unemployment_Rate,Personal_Savings_Rate
2015-04-01,0.12,236.222,59.3,62.8,5.4,5.9
2015-05-01,0.12,237.001,59.4,62.9,5.6,5.8
2015-06-01,0.13,237.657,59.4,62.7,5.3,5.7
2015-07-01,0.13,238.034,59.3,62.6,5.2,5.6
2015-08-01,0.14,238.033,59.4,62.6,5.1,5.6


,Federal_Debt_to_GDP
2015-04-01,99.30094
2015-07-01,98.63595
2015-10-01,102.64192
2016-01-01,103.98904
2016-04-01,103.58005


,T10Y2Y,IGREA
2015-04-01,1.32,-96.726032
2015-04-02,1.37,NaN
2015-04-03,1.36,NaN
2015-04-06,1.41,NaN
2015-04-07,1.37,NaN


In [ ]:

# GEPU (monthly)
gepu_raw = pd.read_excel(GEPU_FILE, sheet_name="Sheet1")

gepu = gepu_raw.dropna(subset=["GEPU_current"]).copy()
gepu["Date"] = pd.to_datetime(
    dict(
        year=gepu["Year"].astype(int),
        month=gepu["Month"].astype(int),
        day=1,
    ),
    errors="coerce"
)
gepu_df = (
    gepu[["Date", "GEPU_current"]]
      .rename(columns={"GEPU_current": "gepu"})
      .dropna(subset=["Date"])
      .sort_values("Date")
      .drop_duplicates(subset=["Date"], keep="last")
      .set_index("Date")
)

# GPR (daily)
gpr_raw = pd.read_excel(GPR_FILE, sheet_name=0, engine="xlrd")
gpr_raw["date"] = pd.to_datetime(gpr_raw["DAY"].astype(str), format="%Y%m%d", errors="coerce")
gpr_df = (
    gpr_raw[["date", "GPRD"]]
      .rename(columns={"GPRD": "gpr_daily"})
      .dropna(subset=["date"])
      .sort_values("date")
      .drop_duplicates(subset=["date"], keep="last")
      .set_index("date")
)
gpr_df = gpr_df.loc[pd.Timestamp(start_date):pd.Timestamp(end_date)]

display(summarize_frame(gepu_df, "gepu_df"))
display(summarize_frame(gpr_df, "gpr_df"))
display(gepu_df.head())
display(gpr_df.head())


,dataset,rows,cols,date_min,date_max,duplicate_index,total_missing_cells
0,gepu_df,347,1,1997-01-01,2025-11-01,0,0


,dataset,rows,cols,date_min,date_max,duplicate_index,total_missing_cells
0,gpr_df,3683,1,2015-04-01,2025-04-30,0,0


,gepu
Date,
1997-01-01,76.059427
1997-02-01,76.286023
1997-03-01,66.810541
1997-04-01,71.716551
1997-05-01,75.229731


,gpr_daily
date,
2015-04-01,138.928131
2015-04-02,113.846565
2015-04-03,150.144714
2015-04-04,165.628387
2015-04-05,116.457458



## Step 3. Synchronize to a common daily timeline

A full daily index is created first.  
Then lower-frequency variables are aligned to that timeline.


In [ ]:
daily_index = pd.date_range(start_date, end_date, freq="D", name="Date")

economic_month_daily = align_to_daily_with_interpolation(economic_month_df, daily_index)
economic_quarter_daily = align_to_daily_with_interpolation(economic_quarter_df, daily_index)
fred_extra_daily = align_to_daily_with_interpolation(fred_extra_df, daily_index)
gepu_daily = align_to_daily_with_interpolation(gepu_df, daily_index)

# GPR is already daily sentiment data; align to the same daily grid.
gpr_daily_aligned = gpr_df.reindex(daily_index).interpolate(method="time", limit_direction="both")

low_frequency_daily = pd.concat(
    [
        economic_month_daily,
        economic_quarter_daily,
        fred_extra_daily,
        gepu_daily,
        gpr_daily_aligned,
    ],
    axis=1
)
low_frequency_daily.index.name = "Date"

display(summarize_frame(low_frequency_daily, "low_frequency_daily"))
display(low_frequency_daily.head())
display(low_frequency_daily.tail())

,dataset,rows,cols,date_min,date_max,duplicate_index,total_missing_cells
0,low_frequency_daily,3683,11,2015-04-01,2025-04-30,0,0


,Federal_Funds_Rate,CPI,Employment_Pop_Ratio,Labour_Force_Participation_Rate,Unemployment_Rate,Personal_Savings_Rate,Federal_Debt_to_GDP,T10Y2Y,IGREA,gepu,gpr_daily
Date,,,,,,,,,,,
2015-04-01,0.12,236.222000,59.300000,62.800000,5.400000,5.900000,99.300940,1.320000,-96.726032,100.978142,138.928131
2015-04-02,0.12,236.247967,59.303333,62.803333,5.406667,5.896667,99.293632,1.370000,-96.699724,101.094274,113.846565
2015-04-03,0.12,236.273933,59.306667,62.806667,5.413333,5.893333,99.286325,1.360000,-96.673416,101.210406,150.144714
2015-04-04,0.12,236.299900,59.310000,62.810000,5.420000,5.890000,99.279017,1.376667,-96.647108,101.326538,165.628387
2015-04-05,0.12,236.325867,59.313333,62.813333,5.426667,5.886667,99.271710,1.393333,-96.620800,101.442670,116.457458


,Federal_Funds_Rate,CPI,Employment_Pop_Ratio,Labour_Force_Participation_Rate,Unemployment_Rate,Personal_Savings_Rate,Federal_Debt_to_GDP,T10Y2Y,IGREA,gepu,gpr_daily
Date,,,,,,,,,,,
2025-04-26,4.33,320.302,60.0,62.6,4.2,5.5,118.78171,0.553333,-23.563474,623.352045,123.449730
2025-04-27,4.33,320.302,60.0,62.6,4.2,5.5,118.78171,0.556667,-23.563474,623.352045,154.792877
2025-04-28,4.33,320.302,60.0,62.6,4.2,5.5,118.78171,0.560000,-23.563474,623.352045,137.450943
2025-04-29,4.33,320.302,60.0,62.6,4.2,5.5,118.78171,0.540000,-23.563474,623.352045,107.072899
2025-04-30,4.33,320.302,60.0,62.6,4.2,5.5,118.78171,0.570000,-23.563474,623.352045,149.097961



## Step 4. Handle missing values appropriately

### Low-frequency variables
These are interpolated to daily frequency.

### Market-traded series
These are **not** blindly interpolated.  
Instead, missing values are audited against the **COMEX trading calendar**.


In [ ]:
# Build COMEX calendar objects used in the audit and holiday review
target_calendar = xcals.get_calendar(TARGET_CALENDAR_NAME)
comex_sessions = target_calendar.sessions_in_range(start_date, end_date)
comex_non_market_days = daily_index.difference(comex_sessions)

(
    comex_full_closure_holidays,
    comex_special_closes,
    comex_special_opens,
    comex_manual_holiday_overrides,
    comex_holiday_schedule,
    comex_holiday_dates_only,
) = build_calendar_holiday_tables(
    target_calendar,
    start_date,
    end_date,
    manual_overrides=COMEX_HOLIDAY_MANUAL_OVERRIDES,
)

print(f"COMEX sessions between {start_date.date()} and {end_date.date()}: {len(comex_sessions)}")
print(f"Total COMEX non-market days: {len(comex_non_market_days)}")
print(f"Total COMEX holiday-related dates: {len(comex_holiday_dates_only)}")
print(f"Full-closure COMEX holidays: {len(comex_full_closure_holidays)}")
print(f"Special-close COMEX holiday sessions: {len(comex_special_closes)}")
print(f"Special-open COMEX holiday sessions: {len(comex_special_opens)}")
print(f"Manual holiday overrides: {len(comex_manual_holiday_overrides)}")

display(comex_full_closure_holidays.head(20))
display(comex_special_closes.head(20))
display(comex_special_opens.head(20))
display(comex_manual_holiday_overrides)
display(comex_holiday_schedule.head(30))

market_missing_audit, unresolved_market_dates = audit_market_missing_vs_calendar(financial_df, comex_sessions)
display(market_missing_audit)
display(unresolved_market_dates.head(20))


COMEX sessions between 2015-04-01 and 2025-04-30: 2599
Total COMEX non-market days: 1084
Total COMEX holiday-related dates: 111
Full-closure COMEX holidays: 33
Special-close COMEX holiday sessions: 75
Special-open COMEX holiday sessions: 0
Manual holiday overrides: 3


,Date,holiday_name,holiday_type,session_effect,close_time,open_time,tag_source,source_note
0,2015-04-03,Good Friday,full_closure_regular,closed,NaT,NaT,exchange_calendars,Tagged from exchange_calendars holiday rules.
1,2015-12-25,Christmas,full_closure_regular,closed,NaT,NaT,exchange_calendars,Tagged from exchange_calendars holiday rules.
2,2016-01-01,New Year's Day,full_closure_regular,closed,NaT,NaT,exchange_calendars,Tagged from exchange_calendars holiday rules.
3,2016-03-25,Good Friday,full_closure_regular,closed,NaT,NaT,exchange_calendars,Tagged from exchange_calendars holiday rules.
4,2016-12-26,Christmas,full_closure_regular,closed,NaT,NaT,exchange_calendars,Tagged from exchange_calendars holiday rules.
5,2017-01-02,New Year's Day,full_closure_regular,closed,NaT,NaT,exchange_calendars,Tagged from exchange_calendars holiday rules.
6,2017-04-14,Good Friday,full_closure_regular,closed,NaT,NaT,exchange_calendars,Tagged from exchange_calendars holiday rules.
7,2017-12-25,Christmas,full_closure_regular,closed,NaT,NaT,exchange_calendars,Tagged from exchange_calendars holiday rules.
8,2018-01-01,New Year's Day,full_closure_regular,closed,NaT,NaT,exchange_calendars,Tagged from exchange_calendars holiday rules.
9,2018-03-30,Good Friday,full_closure_regular,closed,NaT,NaT,exchange_calendars,Tagged from exchange_calendars holiday rules.


,Date,holiday_name,holiday_type,session_effect,close_time,open_time,tag_source,source_note
0,2015-05-25,Memorial Day,special_close,special_close,12:00:00,NaT,exchange_calendars,Tagged from exchange_calendars holiday rules.
1,2015-07-03,July 4th,special_close,special_close,12:00:00,NaT,exchange_calendars,Tagged from exchange_calendars holiday rules.
2,2015-09-07,Labor Day,special_close,special_close,12:00:00,NaT,exchange_calendars,Tagged from exchange_calendars holiday rules.
3,2015-11-26,Thanksgiving Day,special_close,special_close,12:00:00,NaT,exchange_calendars,Tagged from exchange_calendars holiday rules.
4,2015-11-27,Black Friday,special_close,special_close,12:00:00,NaT,exchange_calendars,Tagged from exchange_calendars holiday rules.
5,2015-12-24,Christmas Eve,special_close,special_close,12:00:00,NaT,exchange_calendars,Tagged from exchange_calendars holiday rules.
6,2016-01-18,Dr. Martin Luther King Jr. Day,special_close,special_close,12:00:00,NaT,exchange_calendars,Tagged from exchange_calendars holiday rules.
7,2016-02-15,Washington's Birthday,special_close,special_close,12:00:00,NaT,exchange_calendars,Tagged from exchange_calendars holiday rules.
8,2016-05-30,Memorial Day,special_close,special_close,12:00:00,NaT,exchange_calendars,Tagged from exchange_calendars holiday rules.
9,2016-07-04,July 4th,special_close,special_close,12:00:00,NaT,exchange_calendars,Tagged from exchange_calendars holiday rules.


,Date,holiday_name,holiday_type,session_effect,close_time,open_time,tag_source,source_note


,Date,holiday_name,holiday_type,session_effect,close_time,open_time,tag_source,source_note
0,2022-06-20,Juneteenth National Independence Day,manual_override,holiday_schedule_notice,NaT,NaT,manual_cme_notice,Juneteenth holiday schedule manually added bec...
1,2023-06-19,Juneteenth National Independence Day,manual_override,holiday_schedule_notice,NaT,NaT,manual_cme_notice,Juneteenth holiday schedule manually added bec...
2,2024-06-19,Juneteenth National Independence Day,manual_override,holiday_schedule_notice,NaT,NaT,manual_cme_notice,Juneteenth holiday schedule manually added bec...


,Date,holiday_name,holiday_type,session_effect,close_time,open_time,tag_source,source_note
0,2015-04-03,Good Friday,full_closure_regular,closed,NaN,NaN,exchange_calendars,Tagged from exchange_calendars holiday rules.
1,2015-05-25,Memorial Day,special_close,special_close,12:00:00,NaN,exchange_calendars,Tagged from exchange_calendars holiday rules.
2,2015-07-03,July 4th,special_close,special_close,12:00:00,NaN,exchange_calendars,Tagged from exchange_calendars holiday rules.
3,2015-09-07,Labor Day,special_close,special_close,12:00:00,NaN,exchange_calendars,Tagged from exchange_calendars holiday rules.
4,2015-11-26,Thanksgiving Day,special_close,special_close,12:00:00,NaN,exchange_calendars,Tagged from exchange_calendars holiday rules.
5,2015-11-27,Black Friday,special_close,special_close,12:00:00,NaN,exchange_calendars,Tagged from exchange_calendars holiday rules.
6,2015-12-24,Christmas Eve,special_close,special_close,12:00:00,NaN,exchange_calendars,Tagged from exchange_calendars holiday rules.
7,2015-12-25,Christmas,full_closure_regular,closed,NaN,NaN,exchange_calendars,Tagged from exchange_calendars holiday rules.
8,2016-01-01,New Year's Day,full_closure_regular,closed,NaN,NaN,exchange_calendars,Tagged from exchange_calendars holiday rules.
9,2016-01-18,Dr. Martin Luther King Jr. Day,special_close,special_close,12:00:00,NaN,exchange_calendars,Tagged from exchange_calendars holiday rules.


,series,observed_points,target_sessions,missing_on_target_sessions,first_observed,last_observed
12,USD_index,729,2599,1880,2022-07-15,2025-04-30
1,Silver_Futures,2124,2599,491,2015-04-01,2025-04-30
0,Gold_Futures,2238,2599,363,2015-04-01,2025-04-30
7,NASDAQ_100,2536,2599,63,2015-04-01,2025-04-30
8,SnP500,2536,2599,63,2015-04-01,2025-04-30
6,iShares_MSCI_World_ETF,2536,2599,63,2015-04-01,2025-04-30
10,CBOE_VIX,2570,2599,39,2015-04-01,2025-04-30
3,Copper_Futures,2619,2599,24,2015-04-01,2025-04-30
9,UST10Y_Treasury_Yield,2625,2599,10,2015-04-01,2025-04-30
2,Crude_Oil_Futures,2651,2599,3,2015-04-01,2025-04-30


,series,date
1086,CBOE_VIX,2015-05-25
1087,CBOE_VIX,2015-07-03
1088,CBOE_VIX,2015-09-07
1089,CBOE_VIX,2015-11-26
1090,CBOE_VIX,2016-01-18
1091,CBOE_VIX,2016-02-15
1092,CBOE_VIX,2016-05-30
1093,CBOE_VIX,2016-07-04
1094,CBOE_VIX,2016-09-05
1095,CBOE_VIX,2016-11-24


In [ ]:
# Yahoo Finance backfill for unresolved traded-session gaps only
# This does NOT interpolate prices. It only fills dates that are still missing after COMEX-session auditing.
yahoo_backfill_summary = pd.DataFrame(columns=["series", "ticker", "unresolved_before", "filled_count", "unresolved_after", "status"])
yahoo_backfill_details = pd.DataFrame(columns=["series", "date", "ticker", "yahoo_close", "status"])

if USE_YAHOO_BACKFILL and not unresolved_market_dates.empty:
    financial_df_backfilled, yahoo_backfill_summary, yahoo_backfill_details = yahoo_backfill_unresolved_only(
        financial_df,
        unresolved_df=unresolved_market_dates,
        ticker_map=yahoo_tickers,
        start_date=start_date,
        end_date=end_date,
        progress=YAHOO_PROGRESS,
        auto_adjust=YAHOO_AUTO_ADJUST,
    )

    market_missing_audit_after, unresolved_market_dates_after = audit_market_missing_vs_calendar(
        financial_df_backfilled,
        comex_sessions
    )

    print("Yahoo backfill applied to unresolved COMEX-session gaps.")
    display(yahoo_backfill_summary)
    display(yahoo_backfill_details.head(20))
    display(market_missing_audit_after)

    financial_final = financial_df_backfilled.copy()
    market_missing_audit_final = market_missing_audit_after.copy()
    unresolved_market_dates_final = unresolved_market_dates_after.copy()
else:
    if unresolved_market_dates.empty:
        print("No unresolved COMEX-session gaps found. Yahoo backfill was not needed.")
    else:
        print("Yahoo backfill disabled. Using original cleaned market panel.")

    financial_final = financial_df.copy()
    market_missing_audit_final = market_missing_audit.copy()
    unresolved_market_dates_final = unresolved_market_dates.copy()


Yahoo backfill applied to unresolved COMEX-session gaps.


,series,ticker,unresolved_before,filled_count,unresolved_after,status
0,USD_index,DX-Y.NYB,1880,1834,46,ok
1,Silver_Futures,SI=F,491,438,53,ok
2,Gold_Futures,GC=F,363,326,37,ok
3,NASDAQ_100,^NDX,63,0,63,no_fill_found
4,SnP500,^GSPC,63,0,63,no_fill_found
5,iShares_MSCI_World_ETF,URTH,63,0,63,no_fill_found
6,CBOE_VIX,^VIX,39,0,39,no_fill_found
7,Copper_Futures,HG=F,24,0,24,no_fill_found
8,UST10Y_Treasury_Yield,^TNX,10,3,7,ok
9,Crude_Oil_Futures,CL=F,3,0,3,no_fill_found


,series,date,ticker,yahoo_close,status
0,CBOE_VIX,2015-05-25,^VIX,NaN,still_unresolved_after_yahoo
1,CBOE_VIX,2015-07-03,^VIX,NaN,still_unresolved_after_yahoo
2,CBOE_VIX,2015-09-07,^VIX,NaN,still_unresolved_after_yahoo
3,CBOE_VIX,2015-11-26,^VIX,NaN,still_unresolved_after_yahoo
4,CBOE_VIX,2016-01-18,^VIX,NaN,still_unresolved_after_yahoo
5,CBOE_VIX,2016-02-15,^VIX,NaN,still_unresolved_after_yahoo
6,CBOE_VIX,2016-05-30,^VIX,NaN,still_unresolved_after_yahoo
7,CBOE_VIX,2016-07-04,^VIX,NaN,still_unresolved_after_yahoo
8,CBOE_VIX,2016-09-05,^VIX,NaN,still_unresolved_after_yahoo
9,CBOE_VIX,2016-11-24,^VIX,NaN,still_unresolved_after_yahoo


,series,observed_points,target_sessions,missing_on_target_sessions,first_observed,last_observed
7,NASDAQ_100,2536,2599,63,2015-04-01,2025-04-30
8,SnP500,2536,2599,63,2015-04-01,2025-04-30
6,iShares_MSCI_World_ETF,2536,2599,63,2015-04-01,2025-04-30
1,Silver_Futures,2562,2599,53,2015-04-01,2025-04-30
12,USD_index,2563,2599,46,2015-04-01,2025-04-30
10,CBOE_VIX,2570,2599,39,2015-04-01,2025-04-30
0,Gold_Futures,2564,2599,37,2015-04-01,2025-04-30
3,Copper_Futures,2619,2599,24,2015-04-01,2025-04-30
9,UST10Y_Treasury_Yield,2628,2599,7,2015-04-01,2025-04-30
2,Crude_Oil_Futures,2651,2599,3,2015-04-01,2025-04-30



## Step 5. Merge all aligned variables

The daily master panel is created **after** low-frequency alignment and market-data auditing.


In [ ]:

financial_daily_full = financial_final.reindex(daily_index)

master_daily_panel = pd.concat(
    [financial_daily_full, low_frequency_daily],
    axis=1
).sort_index()
master_daily_panel.index.name = "Date"

display(summarize_frame(master_daily_panel, "master_daily_panel"))
display(master_daily_panel.head())
display(master_daily_panel.tail())


,dataset,rows,cols,date_min,date_max,duplicate_index,total_missing_cells
0,master_daily_panel,3683,24,2015-04-01,2025-04-30,0,13508


,Gold_Futures,Silver_Futures,Crude_Oil_Futures,Copper_Futures,Platinum_Futures,Palladium_Futures,iShares_MSCI_World_ETF,NASDAQ_100,SnP500,UST10Y_Treasury_Yield,CBOE_VIX,US30,USD_index,Federal_Funds_Rate,CPI,Employment_Pop_Ratio,Labour_Force_Participation_Rate,Unemployment_Rate,Personal_Savings_Rate,Federal_Debt_to_GDP,T10Y2Y,IGREA,gepu,gpr_daily
Date,,,,,,,,,,,,,,,,,,,,,,,,
2015-04-01,1208.2,17.059,50.09,2.750,1161.20,748.85,61.04,4311.26,2059.7,1.859,15.11,17698.2,98.190002,0.12,236.222000,59.300000,62.800000,5.400000,5.900000,99.300940,1.320000,-96.726032,100.978142,138.928131
2015-04-02,1200.9,16.701,49.14,2.736,1155.50,746.30,61.51,4316.01,2067.0,1.913,14.67,17763.2,97.440002,0.12,236.247967,59.303333,62.803333,5.406667,5.896667,99.293632,1.370000,-96.699724,101.094274,113.846565
2015-04-03,NaN,16.720,49.14,2.736,NaN,747.90,NaN,NaN,NaN,1.842,NaN,17676.0,NaN,0.12,236.273933,59.306667,62.806667,5.413333,5.893333,99.286325,1.360000,-96.673416,101.210406,150.144714
2015-04-04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.12,236.299900,59.310000,62.810000,5.420000,5.890000,99.279017,1.376667,-96.647108,101.326538,165.628387
2015-04-05,NaN,NaN,NaN,NaN,1174.85,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.12,236.325867,59.313333,62.813333,5.426667,5.886667,99.271710,1.393333,-96.620800,101.442670,116.457458


,Gold_Futures,Silver_Futures,Crude_Oil_Futures,Copper_Futures,Platinum_Futures,Palladium_Futures,iShares_MSCI_World_ETF,NASDAQ_100,SnP500,UST10Y_Treasury_Yield,CBOE_VIX,US30,USD_index,Federal_Funds_Rate,CPI,Employment_Pop_Ratio,Labour_Force_Participation_Rate,Unemployment_Rate,Personal_Savings_Rate,Federal_Debt_to_GDP,T10Y2Y,IGREA,gepu,gpr_daily
Date,,,,,,,,,,,,,,,,,,,,,,,,
2025-04-26,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.33,320.302,60.0,62.6,4.2,5.5,118.78171,0.553333,-23.563474,623.352045,123.449730
2025-04-27,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,39977.5,NaN,4.33,320.302,60.0,62.6,4.2,5.5,118.78171,0.556667,-23.563474,623.352045,154.792877
2025-04-28,3333.0,33.165,62.05,4.8645,993.1,946.1,534.95,19427.29,5528.75,4.205,25.15,40255.0,99.01,4.33,320.302,60.0,62.6,4.2,5.5,118.78171,0.560000,-23.563474,623.352045,137.450943
2025-04-29,3333.6,33.577,60.42,4.8725,985.5,936.7,540.46,19544.95,5560.83,4.171,24.17,40520.0,99.14,4.33,320.302,60.0,62.6,4.2,5.5,118.78171,0.540000,-23.563474,623.352045,107.072899
2025-04-30,3319.1,32.828,58.21,4.6090,969.4,934.4,545.11,19571.02,5569.06,4.167,24.70,40798.4,99.66,4.33,320.302,60.0,62.6,4.2,5.5,118.78171,0.570000,-23.563474,623.352045,149.097961



## Step 6. Restrict the final modeling panel to valid target trading dates

Because the dependent variable is **gold / silver futures**, the final modeling panel is restricted to **COMEX sessions**.


In [ ]:
panel_comex = master_daily_panel.reindex(comex_sessions).copy()
panel_comex.index.name = "Date"

display(summarize_frame(panel_comex, "panel_comex"))
display(panel_comex.head())
display(panel_comex.tail())

print("Missing values on COMEX panel:")
missing_count = panel_comex.isna().sum().sort_values(ascending=False).to_frame("missing_count")
display(missing_count)

print("COMEX market days Dataframe shape", panel_comex.shape)

total_non_market_days = len(comex_non_market_days)
print("Total COMEX non-market days:", total_non_market_days)
print("Total COMEX holiday-related dates:", len(comex_holiday_dates_only))

total_days = len(pd.date_range(start_date, end_date, freq="D"))
print("Total days from start_date to end_date:", total_days)
print("Total Market days based on Comex calendar:", total_days - total_non_market_days)

print("Gold Non market days shape:",  panel_comex["Gold_Futures"].shape)
print("Silver Non market days shape:", panel_comex["Silver_Futures"].shape)


gold_missing = panel_comex["Gold_Futures"].isna().sum()
silver_missing = panel_comex["Silver_Futures"].isna().sum()

print("Gold missing:", gold_missing)
print("Silver missing:", silver_missing)

gold_data_days = panel_comex["Gold_Futures"].notna().sum()
silver_data_days = panel_comex["Silver_Futures"].notna().sum()

print("GOLD DATA DAYS:", gold_data_days)
print("SILVER DATA DAYS:", silver_data_days)

gold_missing_dates = panel_comex.index[panel_comex["Gold_Futures"].isna()]
silver_missing_dates = panel_comex.index[panel_comex["Silver_Futures"].isna()]

print("\nGold missing dates:", len(gold_missing_dates))
for d in gold_missing_dates:
    print(d.date())

print("\nSilver missing dates:", len(silver_missing_dates))
for d in silver_missing_dates:
    print(d.date())

# Tag missing gold / silver dates against COMEX holiday metadata
comex_gold_missing_holiday_map = annotate_missing_with_comex_holidays(
    gold_missing_dates,
    comex_holiday_schedule,
    "Gold_Futures",
    series_specific_overrides=SERIES_SPECIFIC_MISSING_DATE_OVERRIDES,
)
comex_silver_missing_holiday_map = annotate_missing_with_comex_holidays(
    silver_missing_dates,
    comex_holiday_schedule,
    "Silver_Futures",
    series_specific_overrides=SERIES_SPECIFIC_MISSING_DATE_OVERRIDES,
)

print("\nGold missing dates tagged with COMEX holiday metadata:")
display(comex_gold_missing_holiday_map)

print("\nSilver missing dates tagged with COMEX holiday metadata:")
display(comex_silver_missing_holiday_map)

print("\nGold missing dates still not tagged as holiday-related:")
display(
    comex_gold_missing_holiday_map[
        comex_gold_missing_holiday_map["reason"] == "Not tagged as COMEX holiday metadata"
    ]
)

print("\nSilver missing dates still not tagged as holiday-related:")
display(
    comex_silver_missing_holiday_map[
        comex_silver_missing_holiday_map["reason"] == "Not tagged as COMEX holiday metadata"
    ]
)


,dataset,rows,cols,date_min,date_max,duplicate_index,total_missing_cells
0,panel_comex,2599,24,2015-04-01,2025-04-30,0,404


,Gold_Futures,Silver_Futures,Crude_Oil_Futures,Copper_Futures,Platinum_Futures,Palladium_Futures,iShares_MSCI_World_ETF,NASDAQ_100,SnP500,UST10Y_Treasury_Yield,CBOE_VIX,US30,USD_index,Federal_Funds_Rate,CPI,Employment_Pop_Ratio,Labour_Force_Participation_Rate,Unemployment_Rate,Personal_Savings_Rate,Federal_Debt_to_GDP,T10Y2Y,IGREA,gepu,gpr_daily
Date,,,,,,,,,,,,,,,,,,,,,,,,
2015-04-01,1208.2,17.059,50.09,2.7500,1161.20,748.85,61.04,4311.26,2059.7,1.859,15.11,17698.2,98.190002,0.12,236.222000,59.300000,62.800000,5.400000,5.900000,99.300940,1.32,-96.726032,100.978142,138.928131
2015-04-02,1200.9,16.701,49.14,2.7360,1155.50,746.30,61.51,4316.01,2067.0,1.913,14.67,17763.2,97.440002,0.12,236.247967,59.303333,62.803333,5.406667,5.896667,99.293632,1.37,-96.699724,101.094274,113.846565
2015-04-06,1218.6,17.110,52.14,2.7190,1174.90,768.80,62.12,4350.98,2080.6,1.899,14.74,17880.8,97.110001,0.12,236.351833,59.316667,62.816667,5.433333,5.883333,99.264402,1.41,-96.594492,101.558802,116.789253
2015-04-07,1210.6,16.840,53.98,2.7645,1170.05,769.00,61.28,4344.08,2076.3,1.887,14.78,17875.4,97.830002,0.12,236.377800,59.320000,62.820000,5.440000,5.880000,99.257095,1.37,-96.568184,101.674934,110.125252
2015-04-08,1203.1,16.454,50.42,2.7335,1164.30,755.70,61.83,4375.96,2081.9,1.906,13.98,17902.5,98.070000,0.12,236.403767,59.323333,62.823333,5.446667,5.876667,99.249787,1.38,-96.541876,101.791067,166.755096


,Gold_Futures,Silver_Futures,Crude_Oil_Futures,Copper_Futures,Platinum_Futures,Palladium_Futures,iShares_MSCI_World_ETF,NASDAQ_100,SnP500,UST10Y_Treasury_Yield,CBOE_VIX,US30,USD_index,Federal_Funds_Rate,CPI,Employment_Pop_Ratio,Labour_Force_Participation_Rate,Unemployment_Rate,Personal_Savings_Rate,Federal_Debt_to_GDP,T10Y2Y,IGREA,gepu,gpr_daily
Date,,,,,,,,,,,,,,,,,,,,,,,,
2025-04-24,3348.600000,33.503000,62.79,4.9105,976.8,950.6,530.60,19214.40,5484.77,4.314,26.47,40069.0,99.31,4.33,320.302,60.0,62.6,4.2,5.5,118.78171,0.55,-23.563474,623.352045,179.527267
2025-04-25,3282.399902,32.988998,63.02,4.8950,969.8,933.7,535.36,19432.56,5525.21,4.254,24.84,40126.5,99.58,4.33,320.302,60.0,62.6,4.2,5.5,118.78171,0.55,-23.563474,623.352045,190.112579
2025-04-28,3333.000000,33.165000,62.05,4.8645,993.1,946.1,534.95,19427.29,5528.75,4.205,25.15,40255.0,99.01,4.33,320.302,60.0,62.6,4.2,5.5,118.78171,0.56,-23.563474,623.352045,137.450943
2025-04-29,3333.600000,33.577000,60.42,4.8725,985.5,936.7,540.46,19544.95,5560.83,4.171,24.17,40520.0,99.14,4.33,320.302,60.0,62.6,4.2,5.5,118.78171,0.54,-23.563474,623.352045,107.072899
2025-04-30,3319.100000,32.828000,58.21,4.6090,969.4,934.4,545.11,19571.02,5569.06,4.167,24.70,40798.4,99.66,4.33,320.302,60.0,62.6,4.2,5.5,118.78171,0.57,-23.563474,623.352045,149.097961


Missing values on COMEX panel:


,missing_count
iShares_MSCI_World_ETF,63
NASDAQ_100,63
SnP500,63
Silver_Futures,53
USD_index,46
CBOE_VIX,39
Gold_Futures,37
Copper_Futures,24
UST10Y_Treasury_Yield,7
Platinum_Futures,3


...

,Date,series,holiday_name,holiday_type,session_effect,close_time,open_time,tag_source,source_note,holiday_related,reason
0,2017-01-16,Silver_Futures,Dr. Martin Luther King Jr. Day,special_close,special_close,12:00:00,NaN,exchange_calendars,Tagged from exchange_calendars holiday rules.,True,Dr. Martin Luther King Jr. Day (special_close)
1,2017-02-20,Silver_Futures,Washington's Birthday,special_close,special_close,12:00:00,NaN,exchange_calendars,Tagged from exchange_calendars holiday rules.,True,Washington's Birthday (special_close)
2,2017-05-29,Silver_Futures,Memorial Day,special_close,special_close,12:00:00,NaN,exchange_calendars,Tagged from exchange_calendars holiday rules.,True,Memorial Day (special_close)
3,2017-07-04,Silver_Futures,July 4th,special_close,special_close,12:00:00,NaN,exchange_calendars,Tagged from exchange_calendars holiday rules.,True,July 4th (special_close)
4,2017-09-04,Silver_Futures,Labor Day,special_close,special_close,12:00:00,NaN,exchange_calendars,Tagged from exchange_calendars holiday rules.,True,Labor Day (special_close)
5,2017-11-23,Silver_Futures,Thanksgiving Day,special_close,special_close,12:00:00,NaN,exchange_calendars,Tagged from exchange_calendars holiday rules.,True,Thanksgiving Day (special_close)
6,2018-01-15,Silver_Futures,Dr. Martin Luther King Jr. Day,special_close,special_close,12:00:00,NaN,exchange_calendars,Tagged from exchange_calendars holiday rules.,True,Dr. Martin Luther King Jr. Day (special_close)
7,2018-02-19,Silver_Futures,Washington's Birthday,special_close,special_close,12:00:00,NaN,exchange_calendars,Tagged from exchange_calendars holiday rules.,True,Washington's Birthday (special_close)
8,2018-05-28,Silver_Futures,Memorial Day,special_close,special_close,12:00:00,NaN,exchange_calendars,Tagged from exchange_calendars holiday rules.,True,Memorial Day (special_close)
9,2018-07-04,Silver_Futures,July 4th,special_close,special_close,12:00:00,NaN,exchange_calendars,Tagged from exchange_calendars holiday rules.,True,July 4th (special_close)



Gold missing dates still not tagged as holiday-related:


,Date,series,holiday_name,holiday_type,session_effect,close_time,open_time,tag_source,source_note,holiday_related,reason



Silver missing dates still not tagged as holiday-related:


,Date,series,holiday_name,holiday_type,session_effect,close_time,open_time,tag_source,source_note,holiday_related,reason


## Final gold and silver modeling datasets with COMEX holiday/session features

The original RRL feature lists are preserved, then augmented with COMEX calendar-derived
binary features so the modeling panel can learn holiday-related session effects without
fabricating market prices.

Added holiday/session features:
- `comex_holiday_related`
- `comex_full_closure_holiday`
- `comex_special_close`
- `comex_special_open`
- `is_pre_holiday`
- `is_post_holiday`

The notebook keeps both:
- base RRL datasets using only the original feature lists
- holiday-augmented RRL datasets used as the final `df_gold_RRL` and `df_silver_RRL`


In [ ]:
gold_cols = ["Gold_Futures", "Silver_Futures", "Crude_Oil_Futures",
             "UST10Y_Treasury_Yield", "Federal_Funds_Rate",
             "Employment_Pop_Ratio", "gepu", "gpr_daily"]

silver_cols = ["Silver_Futures", "Gold_Futures", "USD_index",
               "SnP500", "NASDAQ_100", "US30"]

print("Gold-set missing counts:")
print(panel_comex[gold_cols].isna().sum().sort_values(ascending=False))

print("\nSilver-set missing counts:")
print(panel_comex[silver_cols].isna().sum().sort_values(ascending=False))

Gold-set missing counts:
Silver_Futures           53
Gold_Futures             37
UST10Y_Treasury_Yield     7
Crude_Oil_Futures         3
Federal_Funds_Rate        0
Employment_Pop_Ratio      0
gepu                      0
gpr_daily                 0
dtype: int64

Silver-set missing counts:
NASDAQ_100        63
SnP500            63
Silver_Futures    53
USD_index         46
Gold_Futures      37
US30               0
dtype: int64


In [ ]:
def summarize_comex_series(
    series_name,
    panel_comex,
    comex_non_market_days,
    comex_holiday_dates_only,
    comex_holiday_schedule,
    start_date,
    end_date,
    show_dates=True,
    show_tables=True,
):
    """
    Summarize missingness and holiday tagging for one COMEX-panel series.

    Parameters
    ----------
    series_name : str
        Column name in panel_comex, e.g. "Gold_Futures"
    panel_comex : pd.DataFrame
    comex_non_market_days : iterable
    comex_holiday_dates_only : pd.DataFrame or iterable
    comex_holiday_schedule : pd.DataFrame
    start_date, end_date : datetime-like
    show_dates : bool
        Print the actual missing dates
    show_tables : bool
        Display tagged holiday tables
    """

    if series_name not in panel_comex.columns:
        raise ValueError(f"{series_name} not found in panel_comex columns.")

    total_non_market_days = len(comex_non_market_days)
    total_days = len(pd.date_range(start_date, end_date, freq="D"))
    total_market_days = total_days - total_non_market_days

    # Robust holiday-date count
    if isinstance(comex_holiday_dates_only, pd.DataFrame) and "Date" in comex_holiday_dates_only.columns:
        total_holiday_related_days = (
            pd.to_datetime(comex_holiday_dates_only["Date"]).dt.normalize().nunique()
        )
    else:
        total_holiday_related_days = len(pd.to_datetime(comex_holiday_dates_only))

    series_shape = panel_comex[series_name].shape
    missing_count = panel_comex[series_name].isna().sum()
    data_days = panel_comex[series_name].notna().sum()
    missing_dates = panel_comex.index[panel_comex[series_name].isna()]

    print("=" * 100)
    print(f"SERIES = {series_name}")
    print("=" * 100)
    print("COMEX market days DataFrame shape:", panel_comex.shape)
    print("Total COMEX non-market days:", total_non_market_days)
    print("Total COMEX holiday-related dates:", total_holiday_related_days)
    print("Total days from start_date to end_date:", total_days)
    print("Total Market days based on COMEX calendar:", total_market_days)

    print(f"{series_name} shape:", series_shape)
    print(f"{series_name} missing:", missing_count)
    print(f"{series_name} data days:", data_days)

    if show_dates:
        print(f"\n{series_name} missing dates: {len(missing_dates)}")
        for d in missing_dates:
            print(d.date())

    missing_holiday_map = annotate_missing_with_comex_holidays(
        missing_dates,
        comex_holiday_schedule,
        series_name,
        series_specific_overrides=SERIES_SPECIFIC_MISSING_DATE_OVERRIDES,
    )

    if show_tables:
        print(f"\n{series_name} missing dates tagged with COMEX holiday metadata:")
        display(missing_holiday_map)

        print(f"\n{series_name} missing dates still not tagged as holiday-related:")
        if missing_holiday_map.empty or "reason" not in missing_holiday_map.columns:
            print("No missing dates.")
            untagged = pd.DataFrame(columns=missing_holiday_map.columns)
        else:
            untagged = missing_holiday_map[
                missing_holiday_map["reason"] == "Not tagged as COMEX holiday metadata"
            ]
            display(untagged)
    else:
        if missing_holiday_map.empty or "reason" not in missing_holiday_map.columns:
            untagged = pd.DataFrame(columns=missing_holiday_map.columns)
        else:
            untagged = missing_holiday_map[
                missing_holiday_map["reason"] == "Not tagged as COMEX holiday metadata"
            ]

    return {
        "series_name": series_name,
        "shape": series_shape,
        "missing_count": missing_count,
        "data_days": data_days,
        "missing_dates": missing_dates,
        "holiday_map": missing_holiday_map,
        "untagged_missing": untagged,
        "total_non_market_days": total_non_market_days,
        "total_holiday_related_days": total_holiday_related_days,
        "total_days": total_days,
        "total_market_days": total_market_days,
    }

###GOLD gold_summary

In [ ]:
gold_summary = summarize_comex_series(
    series_name="Gold_Futures",
    panel_comex=panel_comex,
    comex_non_market_days=comex_non_market_days,
    comex_holiday_dates_only=comex_holiday_dates_only,
    comex_holiday_schedule=comex_holiday_schedule,
    start_date=start_date,
    end_date=end_date,
)

SERIES = Gold_Futures
COMEX market days DataFrame shape: (2599, 24)
Total COMEX non-market days: 1084
Total COMEX holiday-related dates: 111
Total days from start_date to end_date: 3683
Total Market days based on COMEX calendar: 2599
Gold_Futures shape: (2599,)
Gold_Futures missing: 37
Gold_Futures data days: 2562

Gold_Futures missing dates: 37
2017-05-29
2017-07-04
2017-09-04
2017-11-23
2018-01-15
2018-02-19
2018-05-28
2018-07-04
2018-09-03
2018-11-22
2019-01-21
2019-02-18
2019-05-27
2019-07-04
2019-09-02
2019-11-28
2022-05-30
2022-06-20
2022-07-04
2022-09-05
2022-11-24
2023-01-16
2023-02-20
2023-05-29
2023-06-19
2023-07-04
2023-09-04
2023-11-23
2024-01-15
2024-02-19
2024-05-27
2024-06-19
2024-07-04
2024-09-02
2024-11-28
2025-01-20
2025-02-17

Gold_Futures missing dates tagged with COMEX holiday metadata:


,Date,series,holiday_name,holiday_type,session_effect,close_time,open_time,tag_source,source_note,holiday_related,reason
0,2017-05-29,Gold_Futures,Memorial Day,special_close,special_close,12:00:00,NaN,exchange_calendars,Tagged from exchange_calendars holiday rules.,True,Memorial Day (special_close)
1,2017-07-04,Gold_Futures,July 4th,special_close,special_close,12:00:00,NaN,exchange_calendars,Tagged from exchange_calendars holiday rules.,True,July 4th (special_close)
2,2017-09-04,Gold_Futures,Labor Day,special_close,special_close,12:00:00,NaN,exchange_calendars,Tagged from exchange_calendars holiday rules.,True,Labor Day (special_close)
3,2017-11-23,Gold_Futures,Thanksgiving Day,special_close,special_close,12:00:00,NaN,exchange_calendars,Tagged from exchange_calendars holiday rules.,True,Thanksgiving Day (special_close)
4,2018-01-15,Gold_Futures,Dr. Martin Luther King Jr. Day,special_close,special_close,12:00:00,NaN,exchange_calendars,Tagged from exchange_calendars holiday rules.,True,Dr. Martin Luther King Jr. Day (special_close)
5,2018-02-19,Gold_Futures,Washington's Birthday,special_close,special_close,12:00:00,NaN,exchange_calendars,Tagged from exchange_calendars holiday rules.,True,Washington's Birthday (special_close)
6,2018-05-28,Gold_Futures,Memorial Day,special_close,special_close,12:00:00,NaN,exchange_calendars,Tagged from exchange_calendars holiday rules.,True,Memorial Day (special_close)
7,2018-07-04,Gold_Futures,July 4th,special_close,special_close,12:00:00,NaN,exchange_calendars,Tagged from exchange_calendars holiday rules.,True,July 4th (special_close)
8,2018-09-03,Gold_Futures,Labor Day,special_close,special_close,12:00:00,NaN,exchange_calendars,Tagged from exchange_calendars holiday rules.,True,Labor Day (special_close)
9,2018-11-22,Gold_Futures,Thanksgiving Day,special_close,special_close,12:00:00,NaN,exchange_calendars,Tagged from exchange_calendars holiday rules.,True,Thanksgiving Day (special_close)



Gold_Futures missing dates still not tagged as holiday-related:


,Date,series,holiday_name,holiday_type,session_effect,close_time,open_time,tag_source,source_note,holiday_related,reason


###Silver_summary

In [ ]:
silver_summary = summarize_comex_series(
    series_name="Silver_Futures",
    panel_comex=panel_comex,
    comex_non_market_days=comex_non_market_days,
    comex_holiday_dates_only=comex_holiday_dates_only,
    comex_holiday_schedule=comex_holiday_schedule,
    start_date=start_date,
    end_date=end_date,
)

SERIES = Silver_Futures
COMEX market days DataFrame shape: (2599, 24)
Total COMEX non-market days: 1084
Total COMEX holiday-related dates: 111
Total days from start_date to end_date: 3683
Total Market days based on COMEX calendar: 2599
Silver_Futures shape: (2599,)
Silver_Futures missing: 53
Silver_Futures data days: 2546

Silver_Futures missing dates: 53
2017-01-16
2017-02-20
2017-05-29
2017-07-04
2017-09-04
2017-11-23
2018-01-15
2018-02-19
2018-05-28
2018-07-04
2018-09-03
2018-11-22
2019-01-21
2019-02-18
2019-05-27
2019-07-04
2019-09-02
2019-11-28
2020-01-20
2020-02-17
2020-05-25
2020-07-03
2020-09-07
2020-11-26
2021-01-18
2021-02-15
2021-05-31
2021-07-05
2021-09-06
2021-11-25
2022-01-17
2022-02-21
2022-05-30
2022-06-20
2022-07-04
2022-09-05
2022-11-24
2023-01-16
2023-02-20
2023-05-29
2023-06-19
2023-07-04
2023-09-04
2023-11-23
2024-01-15
2024-02-19
2024-05-27
2024-06-19
2024-07-04
2024-09-02
2024-11-28
2025-01-20
2025-02-17

Silver_Futures missing dates tagged with COMEX holiday met

,Date,series,holiday_name,holiday_type,session_effect,close_time,open_time,tag_source,source_note,holiday_related,reason
0,2017-01-16,Silver_Futures,Dr. Martin Luther King Jr. Day,special_close,special_close,12:00:00,NaN,exchange_calendars,Tagged from exchange_calendars holiday rules.,True,Dr. Martin Luther King Jr. Day (special_close)
1,2017-02-20,Silver_Futures,Washington's Birthday,special_close,special_close,12:00:00,NaN,exchange_calendars,Tagged from exchange_calendars holiday rules.,True,Washington's Birthday (special_close)
2,2017-05-29,Silver_Futures,Memorial Day,special_close,special_close,12:00:00,NaN,exchange_calendars,Tagged from exchange_calendars holiday rules.,True,Memorial Day (special_close)
3,2017-07-04,Silver_Futures,July 4th,special_close,special_close,12:00:00,NaN,exchange_calendars,Tagged from exchange_calendars holiday rules.,True,July 4th (special_close)
4,2017-09-04,Silver_Futures,Labor Day,special_close,special_close,12:00:00,NaN,exchange_calendars,Tagged from exchange_calendars holiday rules.,True,Labor Day (special_close)
5,2017-11-23,Silver_Futures,Thanksgiving Day,special_close,special_close,12:00:00,NaN,exchange_calendars,Tagged from exchange_calendars holiday rules.,True,Thanksgiving Day (special_close)
6,2018-01-15,Silver_Futures,Dr. Martin Luther King Jr. Day,special_close,special_close,12:00:00,NaN,exchange_calendars,Tagged from exchange_calendars holiday rules.,True,Dr. Martin Luther King Jr. Day (special_close)
7,2018-02-19,Silver_Futures,Washington's Birthday,special_close,special_close,12:00:00,NaN,exchange_calendars,Tagged from exchange_calendars holiday rules.,True,Washington's Birthday (special_close)
8,2018-05-28,Silver_Futures,Memorial Day,special_close,special_close,12:00:00,NaN,exchange_calendars,Tagged from exchange_calendars holiday rules.,True,Memorial Day (special_close)
9,2018-07-04,Silver_Futures,July 4th,special_close,special_close,12:00:00,NaN,exchange_calendars,Tagged from exchange_calendars holiday rules.,True,July 4th (special_close)



Silver_Futures missing dates still not tagged as holiday-related:


,Date,series,holiday_name,holiday_type,session_effect,close_time,open_time,tag_source,source_note,holiday_related,reason


###Crude_Oil_Futures_summary

In [ ]:
Crude_Oil_Futures_summary = summarize_comex_series(
    series_name="Crude_Oil_Futures",
    panel_comex=panel_comex,
    comex_non_market_days=comex_non_market_days,
    comex_holiday_dates_only=comex_holiday_dates_only,
    comex_holiday_schedule=comex_holiday_schedule,
    start_date=start_date,
    end_date=end_date,
)

SERIES = Crude_Oil_Futures
COMEX market days DataFrame shape: (2599, 24)
Total COMEX non-market days: 1084
Total COMEX holiday-related dates: 111
Total days from start_date to end_date: 3683
Total Market days based on COMEX calendar: 2599
Crude_Oil_Futures shape: (2599,)
Crude_Oil_Futures missing: 3
Crude_Oil_Futures data days: 2596

Crude_Oil_Futures missing dates: 3
2024-11-28
2025-01-20
2025-02-17

Crude_Oil_Futures missing dates tagged with COMEX holiday metadata:


,Date,series,holiday_name,holiday_type,session_effect,close_time,open_time,tag_source,source_note,holiday_related,reason
0,2024-11-28,Crude_Oil_Futures,Thanksgiving Day,special_close,special_close,12:00:00,NaN,exchange_calendars,Tagged from exchange_calendars holiday rules.,True,Thanksgiving Day (special_close)
1,2025-01-20,Crude_Oil_Futures,Dr. Martin Luther King Jr. Day,special_close,special_close,12:00:00,NaN,exchange_calendars,Tagged from exchange_calendars holiday rules.,True,Dr. Martin Luther King Jr. Day (special_close)
2,2025-02-17,Crude_Oil_Futures,Washington's Birthday,special_close,special_close,12:00:00,NaN,exchange_calendars,Tagged from exchange_calendars holiday rules.,True,Washington's Birthday (special_close)



Crude_Oil_Futures missing dates still not tagged as holiday-related:


,Date,series,holiday_name,holiday_type,session_effect,close_time,open_time,tag_source,source_note,holiday_related,reason


###UST10Y_Treasury_Yield_summary

In [ ]:
UST10Y_Treasury_Yield_summary = summarize_comex_series(
    series_name="UST10Y_Treasury_Yield",
    panel_comex=panel_comex,
    comex_non_market_days=comex_non_market_days,
    comex_holiday_dates_only=comex_holiday_dates_only,
    comex_holiday_schedule=comex_holiday_schedule,
    start_date=start_date,
    end_date=end_date,
)

SERIES = UST10Y_Treasury_Yield
COMEX market days DataFrame shape: (2599, 24)
Total COMEX non-market days: 1084
Total COMEX holiday-related dates: 111
Total days from start_date to end_date: 3683
Total Market days based on COMEX calendar: 2599
UST10Y_Treasury_Yield shape: (2599,)
UST10Y_Treasury_Yield missing: 7
UST10Y_Treasury_Yield data days: 2592

UST10Y_Treasury_Yield missing dates: 7
2015-07-03
2016-11-11
2017-09-04
2020-07-03
2024-11-28
2025-01-20
2025-02-17

UST10Y_Treasury_Yield missing dates tagged with COMEX holiday metadata:


,Date,series,holiday_name,holiday_type,session_effect,close_time,open_time,tag_source,source_note,holiday_related,reason
0,2015-07-03,UST10Y_Treasury_Yield,July 4th,special_close,special_close,12:00:00,NaT,exchange_calendars,Tagged from exchange_calendars holiday rules.,True,July 4th (special_close)
1,2016-11-11,UST10Y_Treasury_Yield,Veterans Day,manual_override,fixed_income_holiday,NaT,NaT,manual_sifma_notice,SIFMA fixed-income market holiday; COMEX commo...,True,Veterans Day (fixed_income_holiday)
2,2017-09-04,UST10Y_Treasury_Yield,Labor Day,special_close,special_close,12:00:00,NaT,exchange_calendars,Tagged from exchange_calendars holiday rules.,True,Labor Day (special_close)
3,2020-07-03,UST10Y_Treasury_Yield,July 4th,special_close,special_close,12:00:00,NaT,exchange_calendars,Tagged from exchange_calendars holiday rules.,True,July 4th (special_close)
4,2024-11-28,UST10Y_Treasury_Yield,Thanksgiving Day,special_close,special_close,12:00:00,NaT,exchange_calendars,Tagged from exchange_calendars holiday rules.,True,Thanksgiving Day (special_close)
5,2025-01-20,UST10Y_Treasury_Yield,Dr. Martin Luther King Jr. Day,special_close,special_close,12:00:00,NaT,exchange_calendars,Tagged from exchange_calendars holiday rules.,True,Dr. Martin Luther King Jr. Day (special_close)
6,2025-02-17,UST10Y_Treasury_Yield,Washington's Birthday,special_close,special_close,12:00:00,NaT,exchange_calendars,Tagged from exchange_calendars holiday rules.,True,Washington's Birthday (special_close)



UST10Y_Treasury_Yield missing dates still not tagged as holiday-related:


,Date,series,holiday_name,holiday_type,session_effect,close_time,open_time,tag_source,source_note,holiday_related,reason


In [ ]:
# Optional check: UST10Y Veterans Day fixed-income override
ust10y_missing_dates = panel_comex.index[panel_comex["UST10Y_Treasury_Yield"].isna()]
comex_ust10y_missing_holiday_map = annotate_missing_with_comex_holidays(
    ust10y_missing_dates,
    comex_holiday_schedule,
    "UST10Y_Treasury_Yield",
    series_specific_overrides=SERIES_SPECIFIC_MISSING_DATE_OVERRIDES,
)

display(
    comex_ust10y_missing_holiday_map[
        comex_ust10y_missing_holiday_map["Date"] == pd.Timestamp("2016-11-11")
    ]
)

,Date,series,holiday_name,holiday_type,session_effect,close_time,open_time,tag_source,source_note,holiday_related,reason
1,2016-11-11,UST10Y_Treasury_Yield,Veterans Day,manual_override,fixed_income_holiday,NaT,NaT,manual_sifma_notice,SIFMA fixed-income market holiday; COMEX commo...,True,Veterans Day (fixed_income_holiday)


###NASDAQ_100_summary

In [ ]:
NASDAQ_100_summary = summarize_comex_series(
    series_name="NASDAQ_100",
    panel_comex=panel_comex,
    comex_non_market_days=comex_non_market_days,
    comex_holiday_dates_only=comex_holiday_dates_only,
    comex_holiday_schedule=comex_holiday_schedule,
    start_date=start_date,
    end_date=end_date,
)

SERIES = NASDAQ_100
COMEX market days DataFrame shape: (2599, 24)
Total COMEX non-market days: 1084
Total COMEX holiday-related dates: 111
Total days from start_date to end_date: 3683
Total Market days based on COMEX calendar: 2599
NASDAQ_100 shape: (2599,)
NASDAQ_100 missing: 63
NASDAQ_100 data days: 2536

NASDAQ_100 missing dates: 63
2015-05-25
2015-07-03
2015-09-07
2015-11-26
2016-01-18
2016-02-15
2016-05-30
2016-07-04
2016-09-05
2016-11-24
2017-01-16
2017-02-20
2017-05-29
2017-07-04
2017-09-04
2017-11-23
2018-01-15
2018-02-19
2018-05-28
2018-07-04
2018-09-03
2018-11-22
2019-01-21
2019-02-18
2019-05-27
2019-07-04
2019-09-02
2019-11-28
2020-01-20
2020-02-17
2020-05-25
2020-07-03
2020-09-07
2020-11-26
2021-01-18
2021-02-15
2021-05-31
2021-07-05
2021-09-06
2021-11-25
2022-01-17
2022-02-21
2022-05-30
2022-06-20
2022-07-04
2022-09-05
2022-11-24
2023-01-16
2023-02-20
2023-05-29
2023-06-19
2023-07-04
2023-09-04
2023-11-23
2024-01-15
2024-02-19
2024-05-27
2024-06-19
2024-07-04
2024-09-02
20

,Date,series,holiday_name,holiday_type,session_effect,close_time,open_time,tag_source,source_note,holiday_related,reason
0,2015-05-25,NASDAQ_100,Memorial Day,special_close,special_close,12:00:00,NaN,exchange_calendars,Tagged from exchange_calendars holiday rules.,True,Memorial Day (special_close)
1,2015-07-03,NASDAQ_100,July 4th,special_close,special_close,12:00:00,NaN,exchange_calendars,Tagged from exchange_calendars holiday rules.,True,July 4th (special_close)
2,2015-09-07,NASDAQ_100,Labor Day,special_close,special_close,12:00:00,NaN,exchange_calendars,Tagged from exchange_calendars holiday rules.,True,Labor Day (special_close)
3,2015-11-26,NASDAQ_100,Thanksgiving Day,special_close,special_close,12:00:00,NaN,exchange_calendars,Tagged from exchange_calendars holiday rules.,True,Thanksgiving Day (special_close)
4,2016-01-18,NASDAQ_100,Dr. Martin Luther King Jr. Day,special_close,special_close,12:00:00,NaN,exchange_calendars,Tagged from exchange_calendars holiday rules.,True,Dr. Martin Luther King Jr. Day (special_close)
...,...,...,...,...,...,...,...,...,...,...,...
58,2024-07-04,NASDAQ_100,July 4th,special_close,special_close,12:00:00,NaN,exchange_calendars,Tagged from exchange_calendars holiday rules.,True,July 4th (special_close)
59,2024-09-02,NASDAQ_100,Labor Day,special_close,special_close,12:00:00,NaN,exchange_calendars,Tagged from exchange_calendars holiday rules.,True,Labor Day (special_close)
60,2024-11-28,NASDAQ_100,Thanksgiving Day,special_close,special_close,12:00:00,NaN,exchange_calendars,Tagged from exchange_calendars holiday rules.,True,Thanksgiving Day (special_close)
61,2025-01-20,NASDAQ_100,Dr. Martin Luther King Jr. Day,special_close,special_close,12:00:00,NaN,exchange_calendars,Tagged from exchange_calendars holiday rules.,True,Dr. Martin Luther King Jr. Day (special_close)



NASDAQ_100 missing dates still not tagged as holiday-related:


,Date,series,holiday_name,holiday_type,session_effect,close_time,open_time,tag_source,source_note,holiday_related,reason


###SnP500_summary

In [ ]:
SnP500_summary = summarize_comex_series(
    series_name="SnP500",
    panel_comex=panel_comex,
    comex_non_market_days=comex_non_market_days,
    comex_holiday_dates_only=comex_holiday_dates_only,
    comex_holiday_schedule=comex_holiday_schedule,
    start_date=start_date,
    end_date=end_date,
)

SERIES = SnP500
COMEX market days DataFrame shape: (2599, 24)
Total COMEX non-market days: 1084
Total COMEX holiday-related dates: 111
Total days from start_date to end_date: 3683
Total Market days based on COMEX calendar: 2599
SnP500 shape: (2599,)
SnP500 missing: 63
SnP500 data days: 2536

SnP500 missing dates: 63
2015-05-25
2015-07-03
2015-09-07
2015-11-26
2016-01-18
2016-02-15
2016-05-30
2016-07-04
2016-09-05
2016-11-24
2017-01-16
2017-02-20
2017-05-29
2017-07-04
2017-09-04
2017-11-23
2018-01-15
2018-02-19
2018-05-28
2018-07-04
2018-09-03
2018-11-22
2019-01-21
2019-02-18
2019-05-27
2019-07-04
2019-09-02
2019-11-28
2020-01-20
2020-02-17
2020-05-25
2020-07-03
2020-09-07
2020-11-26
2021-01-18
2021-02-15
2021-05-31
2021-07-05
2021-09-06
2021-11-25
2022-01-17
2022-02-21
2022-05-30
2022-06-20
2022-07-04
2022-09-05
2022-11-24
2023-01-16
2023-02-20
2023-05-29
2023-06-19
2023-07-04
2023-09-04
2023-11-23
2024-01-15
2024-02-19
2024-05-27
2024-06-19
2024-07-04
2024-09-02
2024-11-28
2025-01-20


,Date,series,holiday_name,holiday_type,session_effect,close_time,open_time,tag_source,source_note,holiday_related,reason
0,2015-05-25,SnP500,Memorial Day,special_close,special_close,12:00:00,NaN,exchange_calendars,Tagged from exchange_calendars holiday rules.,True,Memorial Day (special_close)
1,2015-07-03,SnP500,July 4th,special_close,special_close,12:00:00,NaN,exchange_calendars,Tagged from exchange_calendars holiday rules.,True,July 4th (special_close)
2,2015-09-07,SnP500,Labor Day,special_close,special_close,12:00:00,NaN,exchange_calendars,Tagged from exchange_calendars holiday rules.,True,Labor Day (special_close)
3,2015-11-26,SnP500,Thanksgiving Day,special_close,special_close,12:00:00,NaN,exchange_calendars,Tagged from exchange_calendars holiday rules.,True,Thanksgiving Day (special_close)
4,2016-01-18,SnP500,Dr. Martin Luther King Jr. Day,special_close,special_close,12:00:00,NaN,exchange_calendars,Tagged from exchange_calendars holiday rules.,True,Dr. Martin Luther King Jr. Day (special_close)
...,...,...,...,...,...,...,...,...,...,...,...
58,2024-07-04,SnP500,July 4th,special_close,special_close,12:00:00,NaN,exchange_calendars,Tagged from exchange_calendars holiday rules.,True,July 4th (special_close)
59,2024-09-02,SnP500,Labor Day,special_close,special_close,12:00:00,NaN,exchange_calendars,Tagged from exchange_calendars holiday rules.,True,Labor Day (special_close)
60,2024-11-28,SnP500,Thanksgiving Day,special_close,special_close,12:00:00,NaN,exchange_calendars,Tagged from exchange_calendars holiday rules.,True,Thanksgiving Day (special_close)
61,2025-01-20,SnP500,Dr. Martin Luther King Jr. Day,special_close,special_close,12:00:00,NaN,exchange_calendars,Tagged from exchange_calendars holiday rules.,True,Dr. Martin Luther King Jr. Day (special_close)



SnP500 missing dates still not tagged as holiday-related:


,Date,series,holiday_name,holiday_type,session_effect,close_time,open_time,tag_source,source_note,holiday_related,reason


###Copper_Futures_summary

In [ ]:
Copper_Futures_summary = summarize_comex_series(
    series_name="Copper_Futures",
    panel_comex=panel_comex,
    comex_non_market_days=comex_non_market_days,
    comex_holiday_dates_only=comex_holiday_dates_only,
    comex_holiday_schedule=comex_holiday_schedule,
    start_date=start_date,
    end_date=end_date,
)

SERIES = Copper_Futures
COMEX market days DataFrame shape: (2599, 24)
Total COMEX non-market days: 1084
Total COMEX holiday-related dates: 111
Total days from start_date to end_date: 3683
Total Market days based on COMEX calendar: 2599
Copper_Futures shape: (2599,)
Copper_Futures missing: 24
Copper_Futures data days: 2575

Copper_Futures missing dates: 24
2015-05-25
2015-07-03
2015-09-07
2015-11-26
2016-01-18
2016-02-15
2016-05-30
2016-07-04
2016-09-05
2016-11-24
2017-01-16
2017-02-20
2017-05-29
2017-07-04
2017-09-04
2017-11-23
2018-01-15
2018-02-19
2018-05-28
2018-07-04
2018-09-03
2024-11-28
2025-01-20
2025-02-17

Copper_Futures missing dates tagged with COMEX holiday metadata:


,Date,series,holiday_name,holiday_type,session_effect,close_time,open_time,tag_source,source_note,holiday_related,reason
0,2015-05-25,Copper_Futures,Memorial Day,special_close,special_close,12:00:00,NaN,exchange_calendars,Tagged from exchange_calendars holiday rules.,True,Memorial Day (special_close)
1,2015-07-03,Copper_Futures,July 4th,special_close,special_close,12:00:00,NaN,exchange_calendars,Tagged from exchange_calendars holiday rules.,True,July 4th (special_close)
2,2015-09-07,Copper_Futures,Labor Day,special_close,special_close,12:00:00,NaN,exchange_calendars,Tagged from exchange_calendars holiday rules.,True,Labor Day (special_close)
3,2015-11-26,Copper_Futures,Thanksgiving Day,special_close,special_close,12:00:00,NaN,exchange_calendars,Tagged from exchange_calendars holiday rules.,True,Thanksgiving Day (special_close)
4,2016-01-18,Copper_Futures,Dr. Martin Luther King Jr. Day,special_close,special_close,12:00:00,NaN,exchange_calendars,Tagged from exchange_calendars holiday rules.,True,Dr. Martin Luther King Jr. Day (special_close)
5,2016-02-15,Copper_Futures,Washington's Birthday,special_close,special_close,12:00:00,NaN,exchange_calendars,Tagged from exchange_calendars holiday rules.,True,Washington's Birthday (special_close)
6,2016-05-30,Copper_Futures,Memorial Day,special_close,special_close,12:00:00,NaN,exchange_calendars,Tagged from exchange_calendars holiday rules.,True,Memorial Day (special_close)
7,2016-07-04,Copper_Futures,July 4th,special_close,special_close,12:00:00,NaN,exchange_calendars,Tagged from exchange_calendars holiday rules.,True,July 4th (special_close)
8,2016-09-05,Copper_Futures,Labor Day,special_close,special_close,12:00:00,NaN,exchange_calendars,Tagged from exchange_calendars holiday rules.,True,Labor Day (special_close)
9,2016-11-24,Copper_Futures,Thanksgiving Day,special_close,special_close,12:00:00,NaN,exchange_calendars,Tagged from exchange_calendars holiday rules.,True,Thanksgiving Day (special_close)



Copper_Futures missing dates still not tagged as holiday-related:


,Date,series,holiday_name,holiday_type,session_effect,close_time,open_time,tag_source,source_note,holiday_related,reason


##Gold supplemented by yahoo after non-market dropped

In [ ]:
gold_cols = ["Gold_Futures", "Silver_Futures", "Crude_Oil_Futures",
             "UST10Y_Treasury_Yield", "Federal_Funds_Rate",
             "Employment_Pop_Ratio", "gepu", "gpr_daily"]

gold_extra_loss = panel_comex.loc[
    panel_comex["Gold_Futures"].notna() &
    panel_comex[gold_cols].isna().any(axis=1),
    gold_cols
]

print("Gold extra rows lost:", len(gold_extra_loss))
display(gold_extra_loss)

Gold extra rows lost: 18


,Gold_Futures,Silver_Futures,Crude_Oil_Futures,UST10Y_Treasury_Yield,Federal_Funds_Rate,Employment_Pop_Ratio,gepu,gpr_daily
Date,,,,,,,,
2015-07-03,1173.2,15.753,55.52,NaN,0.130645,59.306452,122.884289,86.954903
2016-11-11,1224.3,17.382,43.41,NaN,0.453333,59.700000,214.145027,94.903030
2017-01-16,1212.9,NaN,52.52,2.378,0.654839,59.948387,217.776470,72.586143
2017-02-20,1238.9,NaN,53.97,2.436,0.748214,60.135714,220.035731,110.418930
2020-01-20,1557.9,NaN,58.73,1.823,1.568387,61.100000,194.646238,119.116302
2020-02-17,1603.6,NaN,52.20,1.578,1.066897,60.382759,263.787934,84.648239
2020-05-25,1705.6,NaN,33.92,0.659,0.073226,54.248387,333.561895,129.133987
2020-07-03,1793.5,NaN,40.28,NaN,0.090645,55.283871,311.611709,43.972137
2020-09-07,1943.2,NaN,39.20,0.720,0.090000,56.780000,308.778625,149.525620


In [ ]:
def compare_target_vs_features(panel_comex, target_col, feature_cols):
    target_missing_dates = set(panel_comex.index[panel_comex[target_col].isna()])

    rows = []

    for feature in feature_cols:
        feature_missing_dates = set(panel_comex.index[panel_comex[feature].isna()])

        overlap = sorted(target_missing_dates & feature_missing_dates)
        feature_only = sorted(feature_missing_dates - target_missing_dates)
        target_only = sorted(target_missing_dates - feature_missing_dates)

        rows.append({
            "target": target_col,
            "feature": feature,
            "target_missing_count": len(target_missing_dates),
            "feature_missing_count": len(feature_missing_dates),
            "overlap_missing_count": len(overlap),
            "feature_only_missing_count": len(feature_only),
            "target_only_missing_count": len(target_only),
        })

    overlap_df = pd.DataFrame(rows).sort_values(
        ["overlap_missing_count", "feature_missing_count"],
        ascending=[False, False]
    )

    display(overlap_df)
    return overlap_df

In [ ]:
gold_overlap_df = compare_target_vs_features(
    panel_comex=panel_comex,
    target_col="Gold_Futures",
    feature_cols=[c for c in gold_cols if c != "Gold_Futures"]
)

,target,feature,target_missing_count,feature_missing_count,overlap_missing_count,feature_only_missing_count,target_only_missing_count
0,Gold_Futures,Silver_Futures,37,53,37,16,0
2,Gold_Futures,UST10Y_Treasury_Yield,37,7,4,3,33
1,Gold_Futures,Crude_Oil_Futures,37,3,3,0,34
3,Gold_Futures,Federal_Funds_Rate,37,0,0,0,37
4,Gold_Futures,Employment_Pop_Ratio,37,0,0,0,37
5,Gold_Futures,gepu,37,0,0,0,37
6,Gold_Futures,gpr_daily,37,0,0,0,37


##Silver supplemented by yahoo after non-market dropped

In [ ]:
silver_overlap_df = compare_target_vs_features(
    panel_comex=panel_comex,
    target_col="Silver_Futures",
    feature_cols=[c for c in silver_cols if c != "Silver_Futures"]
)

,target,feature,target_missing_count,feature_missing_count,overlap_missing_count,feature_only_missing_count,target_only_missing_count
2,Silver_Futures,SnP500,53,63,53,10,0
3,Silver_Futures,NASDAQ_100,53,63,53,10,0
0,Silver_Futures,Gold_Futures,53,37,37,0,16
1,Silver_Futures,USD_index,53,46,35,11,18
4,Silver_Futures,US30,53,0,0,0,53


In [ ]:
silver_cols = ["Silver_Futures", "Gold_Futures", "Copper_Futures",
               "SnP500", "NASDAQ_100", "Crude_Oil_Futures"]

silver_extra_loss = panel_comex.loc[
    panel_comex["Silver_Futures"].notna() &
    panel_comex[silver_cols].isna().any(axis=1),
    silver_cols
]

print("Silver extra rows lost:", len(silver_extra_loss))
display(silver_extra_loss)

Silver extra rows lost: 10


,Silver_Futures,Gold_Futures,Copper_Futures,SnP500,NASDAQ_100,Crude_Oil_Futures
Date,,,,,,
2015-05-25,16.746,1186.9,NaN,NaN,NaN,59.85
2015-07-03,15.753,1173.2,NaN,NaN,NaN,55.52
2015-09-07,14.755,1121.0,NaN,NaN,NaN,44.70
2015-11-26,14.048,1056.2,NaN,NaN,NaN,42.59
2016-01-18,14.121,1089.1,NaN,NaN,NaN,30.02
2016-02-15,15.334,1208.2,NaN,NaN,NaN,30.60
2016-05-30,15.994,1217.5,NaN,NaN,NaN,49.51
2016-07-04,19.907,1358.7,NaN,NaN,NaN,48.65
2016-09-05,20.138,1354.0,NaN,NaN,NaN,45.07


## Save outputs

The notebook saves:
- cleaned financial price panel
- low-frequency daily panel
- master daily panel
- COMEX-restricted master panel
- unresolved market missing dates audit
- COMEX holiday tables (full closures, special hours, manual overrides)
- gold and silver missing-date holiday tags
- final gold and silver RRL datasets


In [ ]:
# ============================================================
# Build / rebuild holiday-augmented COMEX panel
# ============================================================
if "HOLIDAY_FEATURE_COLS" not in globals():
    HOLIDAY_FEATURE_COLS = [
        "comex_holiday_related",
        "comex_full_closure_holiday",
        "comex_special_close",
        "comex_special_open",
        "is_pre_holiday",
        "is_post_holiday",
    ]

# ------------------------------------------------------------
# IMPORTANT:
# Use the ACTUAL 2 new FRED columns you requested
# T10Y2Y = 10-Year Treasury Constant Maturity Minus 2-Year Treasury Constant Maturity
# IGREA  = Index of Global Real Economic Activity
# ------------------------------------------------------------
NEW_FRED_SILVER_FEATURES = ["T10Y2Y", "IGREA"]

# Ensure DatetimeIndex
panel_comex = panel_comex.copy()
panel_comex.index = pd.to_datetime(panel_comex.index)

low_frequency_daily = low_frequency_daily.copy()
low_frequency_daily.index = pd.to_datetime(low_frequency_daily.index)

master_daily_panel = master_daily_panel.copy()
master_daily_panel.index = pd.to_datetime(master_daily_panel.index)

# ------------------------------------------------------------
# Make sure the 2 new FRED columns are actually inside panel_comex
# Pull them from panel_comex if already present, otherwise from
# master_daily_panel or low_frequency_daily
# ------------------------------------------------------------
candidate_sources = {
    "panel_comex": panel_comex,
    "master_daily_panel": master_daily_panel,
    "low_frequency_daily": low_frequency_daily,
}

source_name = None
source_df = None

for name, df in candidate_sources.items():
    if all(col in df.columns for col in NEW_FRED_SILVER_FEATURES):
        source_name = name
        source_df = df[NEW_FRED_SILVER_FEATURES].copy()
        break

if source_df is None:
    raise ValueError(
        f"Could not find {NEW_FRED_SILVER_FEATURES} in panel_comex, "
        "master_daily_panel, or low_frequency_daily. "
        "Fetch and merge them upstream first."
    )

source_df = source_df.reindex(panel_comex.index)

for col in NEW_FRED_SILVER_FEATURES:
    panel_comex[col] = source_df[col]

print(f"Integrated extra FRED columns from: {source_name}")
print(panel_comex[NEW_FRED_SILVER_FEATURES].isna().sum())

# ============================================================
# ALWAYS rebuild holiday panel so it includes the new columns
# ============================================================
panel_comex_with_holidays = panel_comex.copy()

holiday_dates = set(pd.to_datetime(comex_holiday_dates_only["Date"]).dt.normalize())
full_closure_dates = set(pd.to_datetime(comex_full_closure_holidays["Date"]).dt.normalize())
special_close_dates = set(pd.to_datetime(comex_special_closes["Date"]).dt.normalize())
special_open_dates = set(pd.to_datetime(comex_special_opens["Date"]).dt.normalize())

panel_dates = pd.to_datetime(panel_comex_with_holidays.index).normalize()

panel_comex_with_holidays["comex_holiday_related"] = panel_dates.isin(holiday_dates).astype(int)
panel_comex_with_holidays["comex_full_closure_holiday"] = panel_dates.isin(full_closure_dates).astype(int)
panel_comex_with_holidays["comex_special_close"] = panel_dates.isin(special_close_dates).astype(int)
panel_comex_with_holidays["comex_special_open"] = panel_dates.isin(special_open_dates).astype(int)

panel_day_list = list(panel_dates)
pre_holiday_dates = set()
post_holiday_dates = set()

for h in sorted(holiday_dates):
    earlier_dates = [d for d in panel_day_list if d < h]
    later_dates = [d for d in panel_day_list if d > h]
    if earlier_dates:
        pre_holiday_dates.add(max(earlier_dates))
    if later_dates:
        post_holiday_dates.add(min(later_dates))

panel_comex_with_holidays["is_pre_holiday"] = panel_dates.isin(pre_holiday_dates).astype(int)
panel_comex_with_holidays["is_post_holiday"] = panel_dates.isin(post_holiday_dates).astype(int)

holiday_name_map = (
    comex_holiday_schedule[["Date", "holiday_name"]]
    .dropna(subset=["Date"])
    .drop_duplicates(subset=["Date"])
    .assign(Date=lambda x: pd.to_datetime(x["Date"]).dt.normalize())
    .set_index("Date")["holiday_name"]
    .to_dict()
)

panel_comex_with_holidays["comex_holiday_name"] = [
    holiday_name_map.get(d.normalize(), "None")
    for d in pd.to_datetime(panel_comex_with_holidays.index)
]

panel_comex_with_holidays.index = pd.to_datetime(panel_comex_with_holidays.index)

# ============================================================
# Protect gold feature list from notebook-state contamination
# ============================================================
wrong_or_extra_cols = {
    "DTWEXBGS",
    "T10Y2Y",
    "IGREA",
    "10Y_2Y_Treasury_Constant_Maturity",
    "Global_Real_Economic_Act",
}

gold_features_base = [c for c in gold_features_RRL if c not in wrong_or_extra_cols]
silver_features_base = [c for c in silver_features_RRL if c not in wrong_or_extra_cols]

# Silver-only variant with +2 new FRED drivers
silver_features_RRL_plus2 = list(dict.fromkeys(silver_features_base + NEW_FRED_SILVER_FEATURES))

# ============================================================
# Build required column lists
# ============================================================
required_gold_cols_base = [target_gold] + gold_features_base
required_silver_cols_base = [target_silver] + silver_features_base
required_silver_cols_base_plus2 = [target_silver] + silver_features_RRL_plus2

gold_features_RRL_with_holidays = gold_features_base + HOLIDAY_FEATURE_COLS
silver_features_RRL_with_holidays = silver_features_base + HOLIDAY_FEATURE_COLS
silver_features_RRL_with_holidays_plus2 = silver_features_RRL_plus2 + HOLIDAY_FEATURE_COLS

required_gold_cols = [target_gold] + gold_features_RRL_with_holidays
required_silver_cols = [target_silver] + silver_features_RRL_with_holidays
required_silver_cols_plus2 = [target_silver] + silver_features_RRL_with_holidays_plus2

missing_gold_cols_base = [c for c in required_gold_cols_base if c not in panel_comex.columns]
missing_silver_cols_base = [c for c in required_silver_cols_base if c not in panel_comex.columns]
missing_silver_cols_base_plus2 = [c for c in required_silver_cols_base_plus2 if c not in panel_comex.columns]

missing_gold_cols = [c for c in required_gold_cols if c not in panel_comex_with_holidays.columns]
missing_silver_cols = [c for c in required_silver_cols if c not in panel_comex_with_holidays.columns]
missing_silver_cols_plus2 = [c for c in required_silver_cols_plus2 if c not in panel_comex_with_holidays.columns]

if missing_gold_cols_base:
    raise ValueError(f"Missing base gold columns: {missing_gold_cols_base}")
if missing_silver_cols_base:
    raise ValueError(f"Missing base silver columns: {missing_silver_cols_base}")
if missing_silver_cols_base_plus2:
    raise ValueError(f"Missing silver + 2 FRED columns in base panel: {missing_silver_cols_base_plus2}")

if missing_gold_cols:
    raise ValueError(f"Missing holiday-augmented gold columns: {missing_gold_cols}")
if missing_silver_cols:
    raise ValueError(f"Missing holiday-augmented silver columns: {missing_silver_cols}")
if missing_silver_cols_plus2:
    raise ValueError(f"Missing holiday-augmented silver + 2 FRED columns: {missing_silver_cols_plus2}")

print("Gold shape", panel_comex[required_gold_cols_base].shape)
print("Silver shape", panel_comex[required_silver_cols_base].shape)
print("Silver + 2 FRED shape", panel_comex[required_silver_cols_base_plus2].shape)
print("Gold target missing count", panel_comex[target_gold].isna().sum())
print("Silver target missing count", panel_comex[target_silver].isna().sum())

# -------------------------------------------------------------------
# Base RRL datasets
# Yahoo supplemented
# Gold / Silver nonmarket days dropped
# Linear interpolated in the features only
# -------------------------------------------------------------------
df_gold_na_drop = panel_comex[required_gold_cols_base].dropna(subset=[target_gold]).sort_index().copy()
df_silver_na_drop = panel_comex[required_silver_cols_base].dropna(subset=[target_silver]).sort_index().copy()
df_silver_na_drop_plus2 = panel_comex[required_silver_cols_base_plus2].dropna(subset=[target_silver]).sort_index().copy()

df_gold_RRL = df_gold_na_drop.copy()
df_silver_RRL = df_silver_na_drop.copy()
df_silver_RRL_plus2 = df_silver_na_drop_plus2.copy()

gold_interp_cols = [c for c in gold_features_base if c in df_gold_RRL.columns]
silver_interp_cols = [c for c in silver_features_base if c in df_silver_RRL.columns]
silver_interp_cols_plus2 = [c for c in silver_features_RRL_plus2 if c in df_silver_RRL_plus2.columns]

df_gold_RRL[gold_interp_cols] = df_gold_RRL[gold_interp_cols].interpolate(
    method="time",
    limit_direction="both"
)
df_silver_RRL[silver_interp_cols] = df_silver_RRL[silver_interp_cols].interpolate(
    method="time",
    limit_direction="both"
)
df_silver_RRL_plus2[silver_interp_cols_plus2] = df_silver_RRL_plus2[silver_interp_cols_plus2].interpolate(
    method="time",
    limit_direction="both"
)

df_gold_RRL = df_gold_RRL.dropna().copy()
df_silver_RRL = df_silver_RRL.dropna().copy()
df_silver_RRL_plus2 = df_silver_RRL_plus2.dropna().copy()

display(df_gold_RRL.head())
display(df_silver_RRL.head())
display(df_silver_RRL_plus2.head())

print("\nFinal df_gold_RRL shape:", df_gold_RRL.shape)
print("\nFinal df_silver_RRL shape:", df_silver_RRL.shape)
print("\nFinal df_silver_RRL_plus2 shape:", df_silver_RRL_plus2.shape)
print("\nFinal df_gold_RRL nan values:", df_gold_RRL.isna().sum())
print("\nFinal df_silver_RRL nan values:", df_silver_RRL.isna().sum())
print("\nFinal df_silver_RRL_plus2 nan values:", df_silver_RRL_plus2.isna().sum())

# -------------------------------------------------------------------
# Base RRL datasets
# Linear interpolated all columns after dropping rows with missing target
# -------------------------------------------------------------------
df_gold_interpolate = panel_comex[required_gold_cols_base].dropna(subset=[target_gold]).sort_index().copy()
df_silver_interpolate = panel_comex[required_silver_cols_base].dropna(subset=[target_silver]).sort_index().copy()
df_silver_interpolate_plus2 = panel_comex[required_silver_cols_base_plus2].dropna(subset=[target_silver]).sort_index().copy()

df_gold_interpolate = df_gold_interpolate.interpolate(
    method="time",
    limit_direction="both"
)
df_silver_interpolate = df_silver_interpolate.interpolate(
    method="time",
    limit_direction="both"
)
df_silver_interpolate_plus2 = df_silver_interpolate_plus2.interpolate(
    method="time",
    limit_direction="both"
)

print("\nFinal df_gold_interpolate shape:", df_gold_interpolate.shape)
print("\nFinal df_silver_interpolate shape:", df_silver_interpolate.shape)
print("\nFinal df_silver_interpolate_plus2 shape:", df_silver_interpolate_plus2.shape)
print("\nFinal df_gold_interpolate nan values:", df_gold_interpolate.isna().sum())
print("\nFinal df_silver_interpolate nan values:\n", df_silver_interpolate.isna().sum())
print("\nFinal df_silver_interpolate_plus2 nan values:\n", df_silver_interpolate_plus2.isna().sum())


artifacts = {
    "financial_quality_report.csv": financial_quality_report,
    "financial_prices_clean.csv": financial_df,
    "financial_prices_final.csv": financial_final,
    "low_frequency_daily.csv": low_frequency_daily,
    "market_missing_audit.csv": market_missing_audit,
    "market_missing_audit_final.csv": market_missing_audit_final,
    "unresolved_market_dates_final.csv": unresolved_market_dates_final,
    "comex_full_closure_holidays.csv": comex_full_closure_holidays,
    "comex_special_closes.csv": comex_special_closes,
    "comex_special_opens.csv": comex_special_opens,
    "comex_manual_holiday_overrides.csv": comex_manual_holiday_overrides,
    "comex_holiday_schedule.csv": comex_holiday_schedule,
    "comex_holiday_dates_only.csv": comex_holiday_dates_only,
    "gold_missing_dates_with_comex_holidays.csv": comex_gold_missing_holiday_map,
    "silver_missing_dates_with_comex_holidays.csv": comex_silver_missing_holiday_map,
    "master_daily_panel.csv": master_daily_panel,
    "panel_comex.csv": panel_comex,
    "panel_comex_with_holidays.csv": panel_comex_with_holidays,
    "gold_RRL.csv": df_gold_RRL,
    "silver_RRL.csv": df_silver_RRL,
    "silver_RRL_plus2.csv": df_silver_RRL_plus2,
    "gold_RRL_interpolate.csv": df_gold_interpolate,
    "silver_RRL_interpolate.csv": df_silver_interpolate,
    "silver_RRL_interpolate_plus2.csv": df_silver_interpolate_plus2,
}

if SAVE_BACKFILL_ARTIFACTS:
    artifacts["yahoo_backfill_summary.csv"] = yahoo_backfill_summary
    artifacts["yahoo_backfill_details.csv"] = yahoo_backfill_details

for filename, df in artifacts.items():
    out_path = os.path.join(OUTPUT_DIR, filename)
    df.to_csv(out_path, index=True)
    print(f"Saved: {out_path}")

Integrated extra FRED columns from: panel_comex
T10Y2Y    0
IGREA     0
dtype: int64
Gold shape (2599, 8)
Silver shape (2599, 6)
Silver + 2 FRED shape (2599, 8)
Gold target missing count 37
Silver target missing count 53


,Gold_Futures,Silver_Futures,Crude_Oil_Futures,UST10Y_Treasury_Yield,Federal_Funds_Rate,Employment_Pop_Ratio,gepu,gpr_daily
Date,,,,,,,,
2015-04-01,1208.2,17.059,50.09,1.859,0.12,59.300000,100.978142,138.928131
2015-04-02,1200.9,16.701,49.14,1.913,0.12,59.303333,101.094274,113.846565
2015-04-06,1218.6,17.110,52.14,1.899,0.12,59.316667,101.558802,116.789253
2015-04-07,1210.6,16.840,53.98,1.887,0.12,59.320000,101.674934,110.125252
2015-04-08,1203.1,16.454,50.42,1.906,0.12,59.323333,101.791067,166.755096


,Silver_Futures,Gold_Futures,US30,SnP500,NASDAQ_100,USD_index
Date,,,,,,
2015-04-01,17.059,1208.2,17698.2,2059.7,4311.26,98.190002
2015-04-02,16.701,1200.9,17763.2,2067.0,4316.01,97.440002
2015-04-06,17.110,1218.6,17880.8,2080.6,4350.98,97.110001
2015-04-07,16.840,1210.6,17875.4,2076.3,4344.08,97.830002
2015-04-08,16.454,1203.1,17902.5,2081.9,4375.96,98.070000


,Silver_Futures,Gold_Futures,US30,SnP500,NASDAQ_100,USD_index,T10Y2Y,IGREA
Date,,,,,,,,
2015-04-01,17.059,1208.2,17698.2,2059.7,4311.26,98.190002,1.32,-96.726032
2015-04-02,16.701,1200.9,17763.2,2067.0,4316.01,97.440002,1.37,-96.699724
2015-04-06,17.110,1218.6,17880.8,2080.6,4350.98,97.110001,1.41,-96.594492
2015-04-07,16.840,1210.6,17875.4,2076.3,4344.08,97.830002,1.37,-96.568184
2015-04-08,16.454,1203.1,17902.5,2081.9,4375.96,98.070000,1.38,-96.541876



Final df_gold_RRL shape: (2562, 8)

Final df_silver_RRL shape: (2546, 6)

Final df_silver_RRL_plus2 shape: (2546, 8)

Final df_gold_RRL nan values: Gold_Futures             0
Silver_Futures           0
Crude_Oil_Futures        0
UST10Y_Treasury_Yield    0
Federal_Funds_Rate       0
Employment_Pop_Ratio     0
gepu                     0
gpr_daily                0
dtype: int64

Final df_silver_RRL nan values: Silver_Futures    0
Gold_Futures      0
US30              0
SnP500            0
NASDAQ_100        0
USD_index         0
dtype: int64

Final df_silver_RRL_plus2 nan values: Silver_Futures    0
Gold_Futures      0
US30              0
SnP500            0
NASDAQ_100        0
USD_index         0
T10Y2Y            0
IGREA             0
dtype: int64

Final df_gold_interpolate shape: (2562, 8)

Final df_silver_interpolate shape: (2546, 6)

Final df_silver_interpolate_plus2 shape: (2546, 8)

Final df_gold_interpolate nan values: Gold_Futures             0
Silver_Futures           0
Crude_Oil

## Notes

- This notebook intentionally **does not interpolate market-traded prices**.
- Yahoo Finance backfill is now applied by default **only** to unresolved dates that remain missing on valid **COMEX sessions**.
- COMEX holiday review is split into **full closures**, **special closes**, **special opens**, and **manual overrides**.
- The Juneteenth dates **2022-06-20**, **2023-06-19**, and **2024-06-19** are injected as **manual CME holiday-notice overrides** because some `exchange_calendars` environments do not surface them through calendar metadata.
- Backfilled rows are written to `yahoo_backfill_summary.csv` and `yahoo_backfill_details.csv` so you can audit exactly what changed.
- `gold_TRUE`, `gold_MIX`, `silver_TRUE`, and `silver_MIX` were not recreated here because they were **not fully defined in the original script**.  
  Re-add them only after the feature-selection stage is finalized.
